In [ ]:
# --- CELL 1: UNPACK SELF-CONTAINED PROJECT OVERRIDE V3.0 MODULES ---
import json, os, sys
from pathlib import Path

EMBEDDED_FILES = {
  "project_override/__init__.py": "\"\"\"Project Override v3.0: Executable World-Model, Retrodiction, Bayesian Exploration & REPL Agent for ARC-AGI-3.\"\"\"\n__version__ = \"3.0.0\"\n",
  "project_override/config.py": "from dataclasses import dataclass, field\nimport os\nfrom typing import List\n\n@dataclass\nclass AgentConfig:\n    total_kaggle_budget_s: float = float(os.environ.get(\"ARC_TOTAL_BUDGET_S\", \"31500.0\"))\n    expected_num_games: int = int(os.environ.get(\"ARC_EXPECTED_GAMES\", \"100\"))\n    max_actions_per_game: int = int(os.environ.get(\"ARC_MAX_ACTIONS_PER_GAME\", \"450\"))\n    max_actions_per_level: int = int(os.environ.get(\"ARC_MAX_ACTIONS_PER_LEVEL\", \"120\"))\n    max_repl_plan_len: int = int(os.environ.get(\"ARC_MAX_REPL_PLAN_LEN\", \"14\"))\n    llm_call_interval: int = int(os.environ.get(\"ARC_LLM_INTERVAL\", \"6\"))\n    llm_timeout_s: float = float(os.environ.get(\"ARC_LLM_TIMEOUT_S\", \"8.0\"))\n    llm_max_new_tokens: int = int(os.environ.get(\"ARC_LLM_MAX_TOKENS\", \"384\"))\n    max_rolling_turns: int = int(os.environ.get(\"ARC_MAX_ROLLING_TURNS\", \"10\"))\n    enable_hud_masking: bool = os.environ.get(\"ARC_ENABLE_HUD_MASK\", \"1\") == \"1\"\n    enable_undo_backtrack: bool = os.environ.get(\"ARC_ENABLE_UNDO\", \"1\") == \"1\"\n    enable_action_guards: bool = os.environ.get(\"ARC_ENABLE_GUARDS\", \"1\") == \"1\"\n    enable_dead_signatures: bool = os.environ.get(\"ARC_ENABLE_DEAD_SIG\", \"1\") == \"1\"\n    enable_world_model_bfs: bool = os.environ.get(\"ARC_ENABLE_WM_BFS\", \"1\") == \"1\"\n    enable_bayesian_exploration: bool = os.environ.get(\"ARC_ENABLE_BAYESIAN\", \"1\") == \"1\"\n    enable_avo_supervisor: bool = os.environ.get(\"ARC_ENABLE_SUPERVISOR\", \"1\") == \"1\"\n    enable_pattern_click_solver: bool = os.environ.get(\"ARC_ENABLE_CLICK_SOLVER\", \"1\") == \"1\"\n    enable_sokoban_planner: bool = os.environ.get(\"ARC_ENABLE_SOKOBAN\", \"1\") == \"1\"\n    enable_llm: bool = os.environ.get(\"ARC_ENABLE_LLM\", \"1\") == \"1\"\n    speculative_draft_tokens: int = int(os.environ.get(\"ARC_MTP_TOKENS\", \"3\"))\n    vllm_base_urls: List[str] = field(\n        default_factory=lambda: [\n            os.environ.get(\"ARC_LLM_BASE_URL\", \"http://127.0.0.1:8000/v1\"),\n            \"http://127.0.0.1:11434/v1\",\n            \"http://localhost:11434/v1\",\n            \"http://127.0.0.1:18300/v1\",\n            \"http://127.0.0.1:30000/v1\",\n        ]\n    )\n    candidate_model_paths: List[str] = field(\n        default_factory=lambda: [\n            \"/kaggle/input/intel-qwen3-8-flash-next-w4a16-autoround\",\n            \"/kaggle/input/models/dfranzen/intel-qwen3-8-flash-next-w4a16-autoround\",\n            \"/kaggle/input/qwen3-8-flash-next-nvfp4\",\n            \"/kaggle/input/qwen3-8-27b-fp8\",\n            \"/kaggle/input/qwen3-6-27b-fp8\",\n            \"/kaggle/input/gemma-4-31b\",\n            \"/kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/32b-instruct-awq/1\",\n            \"/kaggle/input/qwen2.5-coder/transformers/32b-instruct-awq/1\",\n            \"/kaggle/input/qwen-2.5-coder-32b-instruct-awq\",\n        ]\n    )\n    candidate_drafter_paths: List[str] = field(\n        default_factory=lambda: [\n            \"/kaggle/input/albucino-qwen3-8-flash-next-drafter\",\n            \"/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter\",\n        ]\n    )\n",
  "project_override/perception/__init__.py": "from project_override.perception.observation_processor import (\n    Entity,\n    DynamicHUDMasker,\n    ObservationProcessor,\n    hungarian_match,\n)\n\n__all__ = [\"Entity\", \"DynamicHUDMasker\", \"ObservationProcessor\", \"hungarian_match\"]\n",
  "project_override/perception/observation_processor.py": "from dataclasses import dataclass\nfrom typing import Any, Dict, List, Optional, Set, Tuple\nimport numpy as np\n\n@dataclass\nclass Entity:\n    id: int\n    color: int\n    pixels: List[Tuple[int, int]]\n    mass: int\n    centroid: Tuple[float, float]\n    bbox: Tuple[int, int, int, int]\n\n    @property\n    def shape_signature(self) -> Tuple[int, int, int, int]:\n        r0, c0, r1, c1 = self.bbox\n        return (self.color, self.mass, r1 - r0 + 1, c1 - c0 + 1)\n\ndef hungarian_match(cost_matrix: np.ndarray) -> List[Tuple[int, int]]:\n    if cost_matrix.size == 0:\n        return []\n    n_rows, n_cols = cost_matrix.shape\n    pairs: List[Tuple[int, int]] = []\n    used_r: Set[int] = set()\n    used_c: Set[int] = set()\n    flat_indices = np.argsort(cost_matrix, axis=None)\n    for idx in flat_indices:\n        r = int(idx // n_cols)\n        c = int(idx % n_cols)\n        if r not in used_r and c not in used_c:\n            if cost_matrix[r, c] < 1e6:\n                pairs.append((r, c))\n                used_r.add(r)\n                used_c.add(c)\n                if len(used_r) == n_rows or len(used_c) == n_cols:\n                    break\n    return pairs\n\nclass DynamicHUDMasker:\n    \"\"\"Infers HUD/status regions purely from statistical change frequency along borders.\"\"\"\n    def __init__(self, shape: Tuple[int, int] = (64, 64), enabled: bool = True):\n        self.shape = shape\n        self.enabled = enabled\n        self.hud_mask = np.zeros(shape, dtype=bool)\n        self.change_counts = np.zeros(shape, dtype=np.int32)\n        self.total_transitions = 0\n\n    def reset_level(self) -> None:\n        self.change_counts.fill(0)\n        self.total_transitions = 0\n        self.hud_mask.fill(False)\n\n    def observe_transition(self, pre_grid: np.ndarray, post_grid: np.ndarray) -> None:\n        if not self.enabled or pre_grid.shape != self.shape or post_grid.shape != self.shape:\n            return\n        diff = pre_grid != post_grid\n        changed_px = int(np.sum(diff))\n        if 0 < changed_px <= 160:\n            self.total_transitions += 1\n            self.change_counts[diff] += 1\n            if self.total_transitions >= 1:\n                h, w = self.shape\n                border_zone = np.zeros((h, w), dtype=bool)\n                border_zone[:6, :] = True\n                border_zone[h - 6 :, :] = True\n                border_zone[:, :6] = True\n                border_zone[:, w - 6 :] = True\n                high_freq = self.change_counts >= 1\n                row_ticks = np.sum(border_zone & high_freq, axis=1)\n                col_ticks = np.sum(border_zone & high_freq, axis=0)\n                mask = border_zone & (\n                    self.change_counts >= max(2, int(0.60 * self.total_transitions))\n                )\n                for r in range(h):\n                    if (r < 6 or r >= h - 6) and row_ticks[r] >= 3:\n                        mask[r, :] = True\n                for c in range(w):\n                    if (c < 6 or c >= w - 6) and col_ticks[c] >= 3:\n                        mask[:, c] = True\n                self.hud_mask = mask\n\n    def mask_grid(self, grid: np.ndarray) -> np.ndarray:\n        arr = np.asarray(grid, dtype=np.int8).copy()\n        if self.enabled and arr.shape == self.hud_mask.shape and np.any(self.hud_mask):\n            arr[self.hud_mask] = 0\n        return arr\n\nclass ObservationProcessor:\n    \"\"\"Normalizes grids, segments connected objects, and summarizes multi-frame transitions.\"\"\"\n    def __init__(self, enable_hud_masking: bool = True) -> None:\n        self.hud_masker = DynamicHUDMasker(enabled=enable_hud_masking)\n\n    def normalize_grid(self, raw_frame: Any) -> np.ndarray:\n        if isinstance(raw_frame, np.ndarray):\n            arr = raw_frame\n        elif hasattr(raw_frame, \"frame\"):\n            arr = np.asarray(raw_frame.frame)\n        elif hasattr(raw_frame, \"data\"):\n            arr = np.asarray(raw_frame.data)\n        else:\n            arr = np.asarray(raw_frame)\n        if arr.ndim == 3 and arr.shape[0] >= 1:\n            arr = arr[-1]\n        if arr.ndim != 2:\n            arr = np.zeros((64, 64), dtype=np.int8)\n        return np.asarray(arr, dtype=np.int8)\n\n    def extract_animation_sequence(self, raw_frame: Any) -> List[np.ndarray]:\n        frames_list: List[np.ndarray] = []\n        raw_seq = None\n        for attr in (\"frame\", \"data\", \"frames\", \"grids\"):\n            if hasattr(raw_frame, attr):\n                raw_seq = getattr(raw_frame, attr)\n                break\n        if raw_seq is None:\n            raw_seq = raw_frame\n        arr = np.asarray(raw_seq)\n        if arr.ndim == 3:\n            for i in range(arr.shape[0]):\n                frames_list.append(np.asarray(arr[i], dtype=np.int8))\n        elif arr.ndim == 2:\n            frames_list.append(np.asarray(arr, dtype=np.int8))\n        if not frames_list:\n            frames_list.append(np.zeros((64, 64), dtype=np.int8))\n        return frames_list\n\n    def induce_background_color(\n        self, grid: np.ndarray, learned_floor_color: Optional[int] = None\n    ) -> int:\n        if learned_floor_color is not None:\n            return int(learned_floor_color)\n        arr = np.asarray(grid, dtype=np.int8)\n        vals, counts = np.unique(arr, return_counts=True)\n        return int(vals[int(np.argmax(counts))]) if len(vals) > 0 else 0\n\n    def extract_entities(\n        self,\n        grid: np.ndarray,\n        bg_color: Optional[int] = None,\n        apply_hud_mask: bool = True,\n    ) -> List[Entity]:\n        arr = self.hud_masker.mask_grid(grid) if apply_hud_mask else np.asarray(grid, dtype=np.int8)\n        h, w = arr.shape\n        if bg_color is None:\n            bg_color = self.induce_background_color(arr)\n        visited = np.zeros((h, w), dtype=bool)\n        if apply_hud_mask and self.hud_masker.enabled and self.hud_masker.hud_mask.shape == (h, w):\n            visited |= self.hud_masker.hud_mask\n        entities: List[Entity] = []\n        ent_id = 0\n        for r in range(h):\n            for c in range(w):\n                if visited[r, c] or int(arr[r, c]) == bg_color:\n                    continue\n                color = int(arr[r, c])\n                stack = [(r, c)]\n                visited[r, c] = True\n                pixels: List[Tuple[int, int]] = []\n                r_min = r_max = r\n                c_min = c_max = c\n                while stack:\n                    cr, cc = stack.pop()\n                    pixels.append((cr, cc))\n                    if cr < r_min:\n                        r_min = cr\n                    if cr > r_max:\n                        r_max = cr\n                    if cc < c_min:\n                        c_min = cc\n                    if cc > c_max:\n                        c_max = cc\n                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n                        nr, nc = cr + dr, cc + dc\n                        if (\n                            0 <= nr < h\n                            and 0 <= nc < w\n                            and not visited[nr, nc]\n                            and int(arr[nr, nc]) == color\n                        ):\n                            visited[nr, nc] = True\n                            stack.append((nr, nc))\n                mass = len(pixels)\n                mean_r = sum(p[0] for p in pixels) / float(mass)\n                mean_c = sum(p[1] for p in pixels) / float(mass)\n                entities.append(\n                    Entity(\n                        id=ent_id,\n                        color=color,\n                        pixels=pixels,\n                        mass=mass,\n                        centroid=(mean_r, mean_c),\n                        bbox=(r_min, c_min, r_max, c_max),\n                    )\n                )\n                ent_id += 1\n        return entities\n\n    def track_entities(\n        self, prev_entities: List[Entity], curr_entities: List[Entity]\n    ) -> List[Tuple[Entity, Entity, Tuple[float, float]]]:\n        if not prev_entities or not curr_entities:\n            return []\n        cost = np.full((len(prev_entities), len(curr_entities)), 1e7, dtype=np.float64)\n        for i, pe in enumerate(prev_entities):\n            for j, ce in enumerate(curr_entities):\n                if pe.color == ce.color and abs(pe.mass - ce.mass) <= max(2, int(0.25 * pe.mass)):\n                    dist = abs(pe.centroid[0] - ce.centroid[0]) + abs(\n                        pe.centroid[1] - ce.centroid[1]\n                    )\n                    if dist <= 24.0:\n                        cost[i, j] = dist + abs(pe.mass - ce.mass) * 2.0\n        matches = hungarian_match(cost)\n        tracked: List[Tuple[Entity, Entity, Tuple[float, float]]] = []\n        for r_idx, c_idx in matches:\n            pe = prev_entities[r_idx]\n            ce = curr_entities[c_idx]\n            dr = ce.centroid[0] - pe.centroid[0]\n            dc = ce.centroid[1] - pe.centroid[1]\n            if abs(dr) > 0.1 or abs(dc) > 0.1:\n                tracked.append((pe, ce, (dr, dc)))\n        return tracked\n\n    def compute_delta_omega(self, prev_grid: np.ndarray, curr_grid: np.ndarray) -> int:\n        a = np.asarray(prev_grid, dtype=np.int8)\n        b = np.asarray(curr_grid, dtype=np.int8)\n        if a.shape != b.shape:\n            return 0\n        diff = a != b\n        if self.hud_masker.enabled and self.hud_masker.hud_mask.shape == diff.shape:\n            diff = diff & (~self.hud_masker.hud_mask)\n        return int(np.sum(diff))\n\n    def summarize_transition(\n        self,\n        prev_grid: np.ndarray,\n        curr_grid: np.ndarray,\n        anim_frames: Optional[List[np.ndarray]] = None,\n    ) -> str:\n        f0 = np.asarray(prev_grid, dtype=np.int8)\n        f1 = np.asarray(curr_grid, dtype=np.int8)\n        if f0.shape != f1.shape:\n            return \"SHAPE_CHANGED\"\n        diff = f0 != f1\n        if self.hud_masker.enabled and self.hud_masker.hud_mask.shape == diff.shape:\n            diff = diff & (~self.hud_masker.hud_mask)\n        rs, cs = np.where(diff)\n        if len(rs) == 0:\n            return \"NO_INTERIOR_CHANGE\"\n        samples = []\n        for idx in range(min(8, len(rs))):\n            r, c = int(rs[idx]), int(cs[idx])\n            samples.append(f\"({r},{c}):{int(f0[r, c])}->{int(f1[r, c])}\")\n        anim_note = f\" anim_len={len(anim_frames)}\" if anim_frames and len(anim_frames) > 1 else \"\"\n        return f\"changed_px={len(rs)}{anim_note} [\" + \", \".join(samples) + \"]\"\n\n    def encode_compact_observation(self, grid: np.ndarray, max_entities: int = 24) -> str:\n        bg = self.induce_background_color(grid)\n        ents = self.extract_entities(grid, bg_color=bg)\n        ents_sorted = sorted(ents, key=lambda e: (e.mass, e.color))[:max_entities]\n        vals, counts = np.unique(grid, return_counts=True)\n        palette_str = \", \".join(f\"c{int(v)}:{int(cnt)}\" for v, cnt in zip(vals, counts))\n        ent_parts = [\n            f\"O{e.id}(c={e.color},m={e.mass},rc=({int(round(e.centroid[0]))},{int(round(e.centroid[1]))}),box={e.bbox})\"\n            for e in ents_sorted\n        ]\n        return f\"bg={bg} | palette=[{palette_str}] | objects({len(ents)}): \" + \"; \".join(ent_parts)\n",
  "project_override/memory/__init__.py": "from project_override.memory.action_ledger import (\n    ActionLedger,\n    TransitionRecord,\n    compute_arena_hash,\n    pack_signature_64,\n)\nfrom project_override.memory.hypothesis_store import (\n    BayesianExplorationTracker,\n    HypothesisStore,\n)\n\n__all__ = [\n    \"ActionLedger\",\n    \"TransitionRecord\",\n    \"compute_arena_hash\",\n    \"pack_signature_64\",\n    \"BayesianExplorationTracker\",\n    \"HypothesisStore\",\n]\n",
  "project_override/memory/action_ledger.py": "from collections import deque\nfrom dataclasses import dataclass\nimport hashlib\nfrom typing import Any, Dict, List, Optional, Set, Tuple\nimport numpy as np\nfrom project_override.perception.observation_processor import DynamicHUDMasker\n\n@dataclass\nclass TransitionRecord:\n    step_idx: int\n    level_idx: int\n    pre_hash: int\n    action_id: int\n    x: int\n    y: int\n    post_hash: int\n    delta_omega: int\n    diff_summary: str\n    pre_grid: Optional[np.ndarray] = None\n    post_grid: Optional[np.ndarray] = None\n\ndef compute_arena_hash(\n    grid: np.ndarray,\n    hud_masker: Optional[DynamicHUDMasker] = None,\n) -> int:\n    \"\"\"Computes a 44-bit BLAKE2b state hash using explicit little-endian byte order.\"\"\"\n    arr = np.asarray(grid, dtype=np.int8)\n    if hud_masker is not None:\n        arr = hud_masker.mask_grid(arr)\n    digest = hashlib.blake2b(np.ascontiguousarray(arr).tobytes(), digest_size=8).digest()\n    return int.from_bytes(digest, \"little\") & ((1 << 44) - 1)\n\ndef pack_signature_64(\n    arena_hash: int,\n    action_id: int,\n    coord_x: int = 0,\n    coord_y: int = 0,\n    level_idx: int = 0,\n) -> int:\n    h44 = arena_hash & ((1 << 44) - 1)\n    lvl4 = level_idx & 0xF\n    cy6 = coord_y & 0x3F\n    cx6 = coord_x & 0x3F\n    act4 = action_id & 0xF\n    return (h44 << 20) | (lvl4 << 16) | (cy6 << 10) | (cx6 << 4) | act4\n\nclass ActionLedger:\n    \"\"\"\n    Tracks state-action-result transitions, dead signatures, object click no-ops,\n    and PRO-LONG programmatic event history (arXiv:2607.20064).\n    \"\"\"\n    INVERSE_MAP = {1: 2, 2: 1, 3: 4, 4: 3}\n\n    def __init__(self, ring_size: int = 40, enable_dead_signatures: bool = True):\n        self.enable_dead_signatures = enable_dead_signatures\n        self.dead_set: Set[int] = set()\n        self.death_set: Set[int] = set()\n        self.dead_object_clicks: Set[Tuple[int, Tuple[int, int, int, int]]] = set()\n        self.ring_buffer: deque = deque(maxlen=ring_size)\n        self.transitions: Dict[int, Dict[Tuple[int, int, int], int]] = {}\n        self.state_grids: Dict[int, np.ndarray] = {}\n        self.state_novelty: Dict[int, int] = {}\n        self.state_visit_counts: Dict[int, int] = {}\n        self.action_fail_counts: Dict[Tuple[int, int], int] = {}\n        self.seen_atoms: Set[Tuple[int, int, int]] = set()\n        self.seen_pairs: Set[Tuple[Tuple[int, int, int], Tuple[int, int, int]]] = set()\n        self.history: List[TransitionRecord] = []\n\n    def reset_level(self, clear_deaths: bool = False) -> None:\n        self.ring_buffer.clear()\n        self.transitions.clear()\n        self.state_grids.clear()\n        self.state_novelty.clear()\n        self.state_visit_counts.clear()\n        self.action_fail_counts.clear()\n        self.seen_atoms.clear()\n        self.seen_pairs.clear()\n        self.dead_object_clicks.clear()\n        self.history.clear()\n        if clear_deaths:\n            self.death_set.clear()\n            self.dead_set.clear()\n\n    def record_death(\n        self,\n        pre_hash: int,\n        action_id: int,\n        coord_x: int = 0,\n        coord_y: int = 0,\n        level_idx: int = 0,\n    ) -> None:\n        sig = pack_signature_64(pre_hash, action_id, coord_x, coord_y, level_idx)\n        self.death_set.add(sig)\n        self.dead_set.add(sig)\n        self.action_fail_counts[(pre_hash, int(action_id))] = 999\n\n    def record_transition(\n        self,\n        pre_hash: int,\n        action_id: int,\n        post_hash: int,\n        delta_omega: int,\n        coord_x: int = 0,\n        coord_y: int = 0,\n        level_idx: int = 0,\n        pre_grid: Optional[np.ndarray] = None,\n        post_grid: Optional[np.ndarray] = None,\n        diff_summary: str = \"\",\n        clicked_obj_sig: Optional[Tuple[int, int, int, int]] = None,\n        step_idx: int = 0,\n    ) -> None:\n        self.state_visit_counts[post_hash] = self.state_visit_counts.get(post_hash, 0) + 1\n        if pre_grid is not None and pre_hash not in self.state_grids:\n            self.state_grids[pre_hash] = pre_grid.copy()\n        if delta_omega == 0 or pre_hash == post_hash:\n            if self.enable_dead_signatures:\n                sig = pack_signature_64(pre_hash, action_id, coord_x, coord_y, level_idx)\n                self.dead_set.add(sig)\n                if int(action_id) == 6 and clicked_obj_sig is not None:\n                    self.dead_object_clicks.add((pre_hash, clicked_obj_sig))\n            key = (pre_hash, int(action_id))\n            self.action_fail_counts[key] = self.action_fail_counts.get(key, 0) + 1\n        else:\n            act_key = (int(action_id), int(coord_x), int(coord_y))\n            if pre_hash not in self.transitions:\n                self.transitions[pre_hash] = {}\n            self.transitions[pre_hash][act_key] = post_hash\n            if post_grid is not None and post_hash not in self.state_grids:\n                self.state_grids[post_hash] = post_grid.copy()\n                self.state_novelty[post_hash] = self.update_novelty(post_grid)\n        self.ring_buffer.append(post_hash)\n        self.history.append(\n            TransitionRecord(\n                step_idx=step_idx,\n                level_idx=level_idx,\n                pre_hash=pre_hash,\n                action_id=int(action_id),\n                x=int(coord_x),\n                y=int(coord_y),\n                post_hash=post_hash,\n                delta_omega=int(delta_omega),\n                diff_summary=diff_summary,\n                pre_grid=pre_grid.copy() if pre_grid is not None else None,\n                post_grid=post_grid.copy() if post_grid is not None else None,\n            )\n        )\n        if len(self.history) > 250:\n            self.history = self.history[-250:]\n\n    def query_history(\n        self, action_id: Optional[int] = None, min_delta: int = 0\n    ) -> List[Dict[str, Any]]:\n        \"\"\"PRO-LONG programmatic search helper over recorded transitions.\"\"\"\n        out: List[Dict[str, Any]] = []\n        for rec in self.history:\n            if action_id is not None and rec.action_id != int(action_id):\n                continue\n            if rec.delta_omega < min_delta:\n                continue\n            out.append(\n                {\n                    \"step\": rec.step_idx,\n                    \"level\": rec.level_idx,\n                    \"action\": rec.action_id,\n                    \"x\": rec.x,\n                    \"y\": rec.y,\n                    \"delta\": rec.delta_omega,\n                    \"diff\": rec.diff_summary,\n                }\n            )\n        return out\n\n    def update_novelty(self, grid: np.ndarray) -> int:\n        arr = np.asarray(grid, dtype=np.int8)\n        new_count = 0\n        active_atoms: List[Tuple[int, int, int]] = []\n        for r in range(0, arr.shape[0], 4):\n            for c in range(0, arr.shape[1], 4):\n                atom = (r, c, int(arr[r, c]))\n                active_atoms.append(atom)\n                if atom not in self.seen_atoms:\n                    self.seen_atoms.add(atom)\n                    new_count += 2\n        if new_count == 0 and len(active_atoms) <= 256 and len(self.seen_pairs) < 10000:\n            for i in range(0, len(active_atoms), 4):\n                for j in range(i + 1, min(i + 5, len(active_atoms))):\n                    pair = (active_atoms[i], active_atoms[j])\n                    if pair not in self.seen_pairs:\n                        self.seen_pairs.add(pair)\n                        new_count += 1\n        return new_count\n\n    def is_dead(\n        self,\n        arena_hash: int,\n        action_id: int,\n        coord_x: int = 0,\n        coord_y: int = 0,\n        level_idx: int = 0,\n        obj_sig: Optional[Tuple[int, int, int, int]] = None,\n    ) -> bool:\n        if not self.enable_dead_signatures:\n            return False\n        sig = pack_signature_64(arena_hash, action_id, coord_x, coord_y, level_idx)\n        if (sig in self.dead_set) or (sig in self.death_set):\n            return True\n        if int(action_id) == 6 and obj_sig is not None:\n            if (arena_hash, obj_sig) in self.dead_object_clicks:\n                return True\n        return False\n\n    def is_in_loop(self) -> bool:\n        if len(self.ring_buffer) < 4:\n            return False\n        recent = list(self.ring_buffer)[-6:]\n        return len(set(recent)) <= 2\n\n    def get_untried_discrete_actions(\n        self, arena_hash: int, available_actions: List[int], level_idx: int = 0\n    ) -> List[int]:\n        tried = {k[0] for k in self.transitions.get(arena_hash, {}).keys()}\n        return [\n            int(a)\n            for a in available_actions\n            if int(a) in (1, 2, 3, 4, 5)\n            and int(a) not in tried\n            and not self.is_dead(arena_hash, int(a), 0, 0, level_idx)\n        ]\n\n    def bfs_to_unexplored_action(\n        self,\n        start_hash: int,\n        available_discrete: List[int],\n        level_idx: int = 0,\n        last_action_id: Optional[int] = None,\n    ) -> Optional[List[int]]:\n        if not available_discrete:\n            return None\n        untried_here = self.get_untried_discrete_actions(\n            start_hash, available_discrete, level_idx\n        )\n        if untried_here:\n            if (\n                last_action_id is not None\n                and len(untried_here) > 1\n                and self.INVERSE_MAP.get(last_action_id) in untried_here\n            ):\n                non_inv = [\n                    a for a in untried_here if a != self.INVERSE_MAP.get(last_action_id)\n                ]\n                return [non_inv[0]]\n            return [untried_here[0]]\n        queue: deque = deque([(start_hash, [])])\n        visited: Set[int] = {start_hash}\n        while queue and len(visited) < 500:\n            curr_h, path = queue.popleft()\n            if len(path) >= 18:\n                continue\n            for (act_id, cx, cy), next_h in self.transitions.get(curr_h, {}).items():\n                if cx != 0 or cy != 0 or act_id not in available_discrete:\n                    continue\n                if self.is_dead(curr_h, act_id, 0, 0, level_idx):\n                    continue\n                new_path = path + [act_id]\n                if self.get_untried_discrete_actions(next_h, available_discrete, level_idx):\n                    nxt_untried = self.get_untried_discrete_actions(\n                        next_h, available_discrete, level_idx\n                    )\n                    return new_path + [nxt_untried[0]]\n                if next_h not in visited:\n                    visited.add(next_h)\n                    queue.append((next_h, new_path))\n        return None\n\n    def compute_action_penalty(\n        self,\n        arena_hash: int,\n        action_id: int,\n        last_action_id: Optional[int] = None,\n        coord_x: int = 0,\n        coord_y: int = 0,\n        level_idx: int = 0,\n    ) -> float:\n        if self.is_dead(arena_hash, action_id, coord_x, coord_y, level_idx):\n            return 1e6\n        penalty = float(self.action_fail_counts.get((arena_hash, int(action_id)), 0) * 20.0)\n        act_key = (int(action_id), int(coord_x), int(coord_y))\n        known_out = self.transitions.get(arena_hash, {})\n        if act_key in known_out:\n            dest_hash = known_out[act_key]\n            visits = self.state_visit_counts.get(dest_hash, 0)\n            novelty = self.state_novelty.get(dest_hash, 0)\n            penalty += visits * 5.0 - min(12.0, novelty * 1.5)\n        if last_action_id is not None and self.INVERSE_MAP.get(last_action_id) == int(action_id):\n            penalty += 3.5\n        return penalty\n",
  "project_override/memory/hypothesis_store.py": "from collections import deque\nimport math\nfrom typing import Dict, List, Optional, Set, Tuple\nimport numpy as np\nfrom project_override.perception.observation_processor import Entity\n\nclass BayesianExplorationTracker:\n    \"\"\"\n    OPINE-World Lifted Bayesian Belief State (arXiv:2607.01531):\n    Tracks object-action-context transition counts C_t(j, e) with symmetric Dirichlet\n    prior alpha_0 = 1.0, computes normalized posterior predictive entropy eta_j,\n    and ranks unresolved object interactions by actor-facing exploration priority:\n        priority(j) = 1.0 - max_e C_t(j, e) / (1.0 + n_j).\n    \"\"\"\n    def __init__(self, alpha_0: float = 1.0, effect_alphabet_size: int = 6):\n        self.alpha_0 = alpha_0\n        self.m = max(2, effect_alphabet_size)\n        self.outcome_counts: Dict[Tuple[Tuple[int, int, int, int], int], Dict[str, int]] = {}\n\n    def record_effect(\n        self, obj_sig: Tuple[int, int, int, int], action_id: int, effect_tag: str\n    ) -> None:\n        key = (obj_sig, int(action_id))\n        if key not in self.outcome_counts:\n            self.outcome_counts[key] = {}\n        bucket = self.outcome_counts[key]\n        bucket[effect_tag] = bucket.get(effect_tag, 0) + 1\n\n    def posterior_entropy(\n        self, obj_sig: Tuple[int, int, int, int], action_id: int\n    ) -> float:\n        key = (obj_sig, int(action_id))\n        counts = self.outcome_counts.get(key, {})\n        n_j = sum(counts.values())\n        denom = self.m * self.alpha_0 + float(n_j)\n        probs: List[float] = []\n        observed_types = list(counts.values())\n        for c_val in observed_types:\n            probs.append((self.alpha_0 + float(c_val)) / denom)\n        unobserved_slots = max(0, self.m - len(observed_types))\n        for _ in range(unobserved_slots):\n            probs.append(self.alpha_0 / denom)\n        norm_sum = sum(probs)\n        if norm_sum <= 0:\n            return 1.0\n        probs = [p / norm_sum for p in probs]\n        ent = -sum(p * math.log(max(1e-12, p)) for p in probs)\n        return float(ent / math.log(float(self.m)))\n\n    def exploration_priority(\n        self, obj_sig: Tuple[int, int, int, int], action_id: int\n    ) -> float:\n        key = (obj_sig, int(action_id))\n        counts = self.outcome_counts.get(key, {})\n        n_j = sum(counts.values())\n        if n_j == 0:\n            return 1.0\n        max_c = max(counts.values())\n        base_prio = 1.0 - (float(max_c) / float(1 + n_j))\n        if len(counts) == 1 and \"NO_OP\" in counts:\n            return base_prio * (0.25 / float(n_j))\n        return base_prio + 0.15 * self.posterior_entropy(obj_sig, action_id)\n\nclass HypothesisStore:\n    \"\"\"\n    S4 Epistemic Persistence Store (arXiv:2610.26632) + Retrodict Playbook Memory:\n    Separates WorkingModel (checked_rules vs assumed_hypotheses across levels)\n    from WorkingMemory (current level attempt state) and retains reasoning replay.\n    \"\"\"\n    def __init__(self, max_reasoning_items: int = 14):\n        self.displacement_vectors: Dict[int, Tuple[int, int]] = {}\n        self.avatar_colors: Set[int] = set()\n        self.wall_colors: Set[int] = set()\n        self.hazard_colors: Set[int] = set()\n        self.goal_colors: Set[int] = set()\n        self.pushable_colors: Set[int] = set()\n        self.switch_colors: Set[int] = set()\n        self.click_color_transitions: Dict[int, int] = {}\n        self.portal_map: Dict[Tuple[int, int], Tuple[int, int]] = {}\n        self.checked_rules: Dict[str, str] = {}\n        self.assumed_hypotheses: Dict[str, str] = {}\n        self.persistent_notes: Dict[str, str] = {}\n        self.reasoning_replay: deque = deque(maxlen=max_reasoning_items)\n        self.level_clear_summaries: List[str] = []\n        self.bayesian_tracker = BayesianExplorationTracker()\n        self.interacted_object_centers: Set[Tuple[int, int]] = set()\n\n    def reset_for_new_level(self, cleared_previous: bool = False, prev_level: int = 0) -> None:\n        if cleared_previous:\n            vec_str = \", \".join(f\"A{k}:{v}\" for k, v in sorted(self.displacement_vectors.items()))\n            summary = (\n                f\"Cleared Level {prev_level}: avatar={sorted(self.avatar_colors)}, \"\n                f\"vectors={{{vec_str}}}, pushable={sorted(self.pushable_colors)}, \"\n                f\"click_map={self.click_color_transitions}\"\n            )\n            self.level_clear_summaries.append(summary)\n            self.checked_rules[f\"L{prev_level}_verified\"] = summary\n        self.portal_map.clear()\n        self.interacted_object_centers.clear()\n\n    def observe_entity_motions(\n        self,\n        action_id: int,\n        tracked: List[Tuple[Entity, Entity, Tuple[float, float]]],\n        pre_grid: Optional[np.ndarray] = None,\n        post_grid: Optional[np.ndarray] = None,\n        click_xy: Tuple[int, int] = (0, 0),\n    ) -> None:\n        if action_id == 6 and pre_grid is not None and post_grid is not None:\n            cx, cy = int(click_xy[0]), int(click_xy[1])\n            if 0 <= cy < pre_grid.shape[0] and 0 <= cx < pre_grid.shape[1]:\n                c_before = int(pre_grid[cy, cx])\n                c_after = int(post_grid[cy, cx])\n                if c_before != c_after:\n                    self.click_color_transitions[c_before] = c_after\n                    self.checked_rules[f\"click_c{c_before}\"] = f\"c{c_before}->c{c_after}\"\n            return\n        if action_id not in (1, 2, 3, 4) or not tracked:\n            return\n\n        if len(tracked) == 1:\n            pe, ce, (dr, dc) = tracked[0]\n            ir, ic = int(round(dr)), int(round(dc))\n            if (ir != 0 or ic != 0) and abs(ir) <= 16 and abs(ic) <= 16:\n                known_strides = [\n                    max(abs(v[0]), abs(v[1])) for v in self.displacement_vectors.values()\n                ]\n                min_stride = min(known_strides) if known_strides else max(abs(ir), abs(ic))\n                if known_strides and max(abs(ir), abs(ic)) > max(3, min_stride * 2):\n                    p_in = (\n                        int(round(pe.centroid[0]))\n                        + self.displacement_vectors.get(action_id, (0, 0))[0],\n                        int(round(pe.centroid[1]))\n                        + self.displacement_vectors.get(action_id, (0, 0))[1],\n                    )\n                    p_out = (int(round(ce.centroid[0])), int(round(ce.centroid[1])))\n                    self.portal_map[p_in] = p_out\n                else:\n                    self.displacement_vectors[int(action_id)] = (ir, ic)\n                    if pe.color not in self.pushable_colors:\n                        self.avatar_colors.add(pe.color)\n                    self.checked_rules[f\"move_A{action_id}\"] = f\"delta=({ir},{ic})\"\n        else:\n            # Multi-entity movement: isolate controllable avatar from displaced pushable crates\n            av_item = None\n            for item in tracked:\n                if self.avatar_colors and item[0].color in self.avatar_colors:\n                    av_item = item\n                    break\n            if av_item is None:\n                non_push = [item for item in tracked if item[0].color not in self.pushable_colors]\n                av_item = non_push[0] if non_push else tracked[0]\n\n            if av_item is not None and av_item[0].color not in self.pushable_colors:\n                self.avatar_colors.add(av_item[0].color)\n\n            for item in tracked:\n                if item is not av_item:\n                    pe_push, ce_push, (dr_p, dc_p) = item\n                    if abs(dr_p) > 0.1 or abs(dc_p) > 0.1:\n                        if pe_push.color not in self.avatar_colors:\n                            self.pushable_colors.add(pe_push.color)\n                            self.checked_rules[f\"pushable_c{pe_push.color}\"] = \"movable_block\"\n\n    def locate_avatar(self, entities: List[Entity]) -> Optional[Tuple[int, int]]:\n        if not entities:\n            return None\n        if self.avatar_colors:\n            candidates = [e for e in entities if e.color in self.avatar_colors]\n            if candidates:\n                best = min(candidates, key=lambda e: e.mass)\n                return (int(round(best.centroid[0])), int(round(best.centroid[1])))\n        smallest = min(entities, key=lambda e: (e.mass, e.id))\n        return (int(round(smallest.centroid[0])), int(round(smallest.centroid[1])))\n\n    def add_reasoning_trace(self, step_idx: int, level_idx: int, thought: str) -> None:\n        clean = thought.strip().replace(\"\\n\", \" \")[:220]\n        if clean:\n            self.reasoning_replay.append(f\"[L{level_idx}:S{step_idx}] {clean}\")\n\n    def format_epistemic_memory(self) -> str:\n        vecs = \", \".join(f\"A{k}:{v}\" for k, v in sorted(self.displacement_vectors.items()))\n        checked_str = \"; \".join(f\"{k}={v}\" for k, v in list(self.checked_rules.items())[-6:])\n        notes_str = \"; \".join(f\"{k}={v}\" for k, v in list(self.persistent_notes.items())[-6:])\n        clears = \" | \".join(self.level_clear_summaries[-4:])\n        replay = \"\\n  \".join(self.reasoning_replay)\n        return (\n            f\"PLAYBOOK_WORKING_MODEL (CHECKED): {{{checked_str or vecs}}}\\n\"\n            f\"AVATAR_COLORS: {sorted(self.avatar_colors)} | PUSHABLE: {sorted(self.pushable_colors)} | CLICK_MAP: {self.click_color_transitions}\\n\"\n            f\"LEVEL_INVARIANTS: {clears or 'None yet'}\\n\"\n            f\"WORKING_MEMORY_NOTES: {notes_str or 'Empty'}\\n\"\n            f\"S4_REASONING_REPLAY:\\n  {replay or 'No prior reasoning recorded.'}\"\n        )\n",
  "project_override/planning/__init__.py": "from project_override.planning.explorer import SmartClickExplorer, bfs_frontier_explore\nfrom project_override.planning.planner import (\n    EmpiricalWorldModel,\n    MacroActionQueue,\n    astar_grid,\n)\n\n__all__ = [\n    \"SmartClickExplorer\",\n    \"bfs_frontier_explore\",\n    \"EmpiricalWorldModel\",\n    \"MacroActionQueue\",\n    \"astar_grid\",\n]\n",
  "project_override/planning/explorer.py": "from collections import deque\nfrom typing import Dict, List, Optional, Set, Tuple\nimport numpy as np\nfrom project_override.memory.action_ledger import ActionLedger\nfrom project_override.memory.hypothesis_store import BayesianExplorationTracker\nfrom project_override.perception.observation_processor import Entity, ObservationProcessor\n\ndef bfs_frontier_explore(\n    start: Tuple[int, int],\n    visited_cells: Set[Tuple[int, int]],\n    walls: Set[Tuple[int, int]],\n    action_vectors: Dict[int, Tuple[int, int]],\n    grid_shape: Tuple[int, int] = (64, 64),\n) -> Optional[List[int]]:\n    if not action_vectors:\n        return None\n    h, w = grid_shape\n    strides = [\n        max(abs(dr), abs(dc))\n        for dr, dc in action_vectors.values()\n        if (dr, dc) != (0, 0)\n    ]\n    s = max(1, min(strides)) if strides else 1\n    queue: deque = deque([(start, [])])\n    seen: Set[Tuple[int, int]] = {start}\n    while queue and len(seen) < 1200:\n        curr, path = queue.popleft()\n        norm_cell = (curr[0] // s, curr[1] // s)\n        if path and norm_cell not in visited_cells:\n            return path\n        if len(path) >= 22:\n            continue\n        for act_id, (dr, dc) in action_vectors.items():\n            if dr == 0 and dc == 0:\n                continue\n            nr, nc = curr[0] + dr, curr[1] + dc\n            if not (0 <= nr < h and 0 <= nc < w):\n                continue\n            steps_n = max(1, max(abs(dr), abs(dc)))\n            blocked = any(\n                (\n                    int(round(curr[0] + (dr * k) / steps_n)),\n                    int(round(curr[1] + (dc * k) / steps_n)),\n                )\n                in walls\n                for k in range(1, steps_n + 1)\n            )\n            if not blocked and (nr, nc) not in seen:\n                seen.add((nr, nc))\n                queue.append(((nr, nc), path + [int(act_id)]))\n    return None\n\nclass SmartClickExplorer:\n    \"\"\"\n    Selects high-information-gain click coordinates using object segmentation,\n    OPINE-World Bayesian exploration priority S_B (arXiv:2607.01531),\n    and Reki dead-signature pruning.\n    \"\"\"\n    def __init__(self, processor: ObservationProcessor):\n        self.processor = processor\n\n    def select_click(\n        self,\n        grid: np.ndarray,\n        arena_hash: int,\n        ledger: ActionLedger,\n        level_idx: int = 0,\n        bayesian_tracker: Optional[BayesianExplorationTracker] = None,\n        enable_bayesian: bool = True,\n    ) -> Tuple[int, Dict[str, int], Optional[Tuple[int, int, int, int]]]:\n        bg = self.processor.induce_background_color(grid)\n        entities = self.processor.extract_entities(grid, bg_color=bg)\n        color_counts: Dict[int, int] = {}\n        for e in entities:\n            color_counts[e.color] = color_counts.get(e.color, 0) + 1\n\n        def _score_entity(ent: Entity) -> Tuple[float, int, int, int]:\n            prio = 1.0\n            if enable_bayesian and bayesian_tracker is not None:\n                prio = bayesian_tracker.exploration_priority(ent.shape_signature, 6)\n            return (-prio, color_counts.get(ent.color, 1), ent.mass, ent.id)\n\n        ranked = sorted(entities, key=_score_entity)\n        for ent in ranked:\n            cy = int(round(ent.centroid[0]))\n            cx = int(round(ent.centroid[1]))\n            if (cy, cx) not in ent.pixels and ent.pixels:\n                cy, cx = ent.pixels[len(ent.pixels) // 2]\n            sig = ent.shape_signature\n            if not ledger.is_dead(arena_hash, 6, cx, cy, level_idx, obj_sig=sig):\n                return 6, {\"x\": int(cx), \"y\": int(cy)}, sig\n        h, w = grid.shape\n        for step in (8, 5, 3):\n            for r in range(step, h - step, step):\n                for c in range(step, w - step, step):\n                    if not ledger.is_dead(arena_hash, 6, c, r, level_idx):\n                        return 6, {\"x\": int(c), \"y\": int(r)}, None\n        return 6, {\"x\": 32, \"y\": 32}, None\n",
  "project_override/planning/planner.py": "from collections import deque\nimport heapq\nfrom typing import Any, Callable, Dict, List, Optional, Set, Tuple\nimport numpy as np\nfrom project_override.memory.action_ledger import TransitionRecord, compute_arena_hash\nfrom project_override.memory.hypothesis_store import HypothesisStore\nfrom project_override.perception.observation_processor import Entity, ObservationProcessor\n\nclass MacroActionQueue:\n    \"\"\"\n    Holds multi-step action sequences generated by the REPL or World-Model planner.\n    Enforces:\n    1) Daniel Franzen's Milestone 2 interior-board change guard (delta_omega == 0 halts plan).\n    2) Retrodict / Schema / Kepler forward-simulation expectation guard (expect_hash mismatch halts plan).\n    \"\"\"\n    def __init__(self) -> None:\n        self.queue: deque = deque()\n        self.active_subagent: str = \"NONE\"\n\n    def push_plan(\n        self, actions: List[Any], subagent_tag: str = \"PLANNER\", max_len: int = 14\n    ) -> None:\n        self.queue.clear()\n        self.active_subagent = subagent_tag\n        for a in actions[:max_len]:\n            if isinstance(a, tuple) and len(a) == 3:\n                self.queue.append((int(a[0]), dict(a[1]), a[2]))\n            elif isinstance(a, tuple) and len(a) == 2:\n                self.queue.append((int(a[0]), dict(a[1]), None))\n            elif isinstance(a, dict) and \"action_id\" in a:\n                aid = int(a[\"action_id\"])\n                payload = (\n                    {\"x\": int(a[\"x\"]), \"y\": int(a[\"y\"])}\n                    if aid == 6 and \"x\" in a and \"y\" in a\n                    else {}\n                )\n                self.queue.append((aid, payload, a.get(\"expect_hash\")))\n            else:\n                self.queue.append((int(a), {}, None))\n\n    def pop_next(\n        self,\n        last_delta_omega: Optional[int] = None,\n        guard_enabled: bool = True,\n        current_hash: Optional[int] = None,\n    ) -> Optional[Tuple[int, Dict[str, Any]]]:\n        if guard_enabled and last_delta_omega == 0:\n            self.clear()\n            return None\n        if self.queue:\n            aid, payload, expected_pre_hash = self.queue[0]\n            if (\n                guard_enabled\n                and expected_pre_hash is not None\n                and current_hash is not None\n                and int(expected_pre_hash) != int(current_hash)\n            ):\n                self.clear()\n                return None\n            self.queue.popleft()\n            return aid, payload\n        self.active_subagent = \"NONE\"\n        return None\n\n    def clear(self) -> None:\n        self.queue.clear()\n        self.active_subagent = \"NONE\"\n\ndef astar_grid(\n    start: Tuple[int, int],\n    target: Tuple[int, int],\n    walls: Set[Tuple[int, int]],\n    action_vectors: Dict[int, Tuple[int, int]],\n    grid_shape: Tuple[int, int] = (64, 64),\n    portals: Optional[Dict[Tuple[int, int], Tuple[int, int]]] = None,\n    max_expansions: int = 2500,\n) -> Optional[List[int]]:\n    if not action_vectors:\n        return None\n    h, w = grid_shape\n    strides = [\n        max(abs(dr), abs(dc))\n        for dr, dc in action_vectors.values()\n        if (dr, dc) != (0, 0)\n    ]\n    s = max(1, min(strides)) if strides else 1\n    if abs(start[0] - target[0]) < s and abs(start[1] - target[1]) < s:\n        return []\n\n    def heuristic(pos: Tuple[int, int]) -> float:\n        d_direct = (abs(pos[0] - target[0]) + abs(pos[1] - target[1])) / float(s)\n        if portals:\n            for p_in, p_out in portals.items():\n                d_portal = (\n                    abs(pos[0] - p_in[0])\n                    + abs(pos[1] - p_in[1])\n                    + abs(p_out[0] - target[0])\n                    + abs(p_out[1] - target[1])\n                ) / float(s)\n                if d_portal + 1.0 < d_direct:\n                    d_direct = d_portal + 1.0\n        return d_direct\n\n    open_heap: List[Tuple[float, int, Tuple[int, int], List[int]]] = [\n        (heuristic(start), 0, start, [])\n    ]\n    visited: Dict[Tuple[int, int], int] = {start: 0}\n    expansions = 0\n    while open_heap and expansions < max_expansions:\n        _, g, curr, path = heapq.heappop(open_heap)\n        expansions += 1\n        if abs(curr[0] - target[0]) < s and abs(curr[1] - target[1]) < s:\n            return path\n        for act_id, (dr, dc) in action_vectors.items():\n            if dr == 0 and dc == 0:\n                continue\n            nr, nc = curr[0] + dr, curr[1] + dc\n            if not (0 <= nr < h and 0 <= nc < w):\n                continue\n            steps_n = max(1, max(abs(dr), abs(dc)))\n            blocked = False\n            for k in range(1, steps_n + 1):\n                mr = int(round(curr[0] + (dr * k) / steps_n))\n                mc = int(round(curr[1] + (dc * k) / steps_n))\n                mid = (mr, mc)\n                is_target_cell = abs(mid[0] - target[0]) < s and abs(mid[1] - target[1]) < s\n                if mid in walls and not is_target_cell:\n                    blocked = True\n                    break\n            if blocked:\n                continue\n            nxt = (nr, nc)\n            if portals:\n                for p_in, p_out in portals.items():\n                    if abs(nxt[0] - p_in[0]) < s and abs(nxt[1] - p_in[1]) < s:\n                        nxt = p_out\n                        break\n            new_g = g + 1\n            if nxt not in visited or new_g < visited[nxt]:\n                visited[nxt] = new_g\n                heapq.heappush(\n                    open_heap,\n                    (new_g + heuristic(nxt), new_g, nxt, path + [int(act_id)]),\n                )\n    return None\n\nclass EmpiricalWorldModel:\n    \"\"\"\n    Executable World Model & Retrodiction Verifier (inspired by Tycho arXiv:2607.28287,\n    Schema schema-harness.github.io, Kepler arXiv:2610.00834, and Retrodict):\n    - Induces transition rules (movement, walls, pushable blocks, portals, click color cycles)\n      purely from observed grids and interaction history.\n    - Backtests (`run_backtest`) candidate models against recorded transitions.\n    - Searches (`plan_interactive_route` / `plan_click_pattern_match`) for minimal-action plans.\n    \"\"\"\n    def __init__(self, processor: ObservationProcessor, hypotheses: HypothesisStore):\n        self.processor = processor\n        self.hypotheses = hypotheses\n        self.custom_step_fn: Optional[Callable[[np.ndarray, int, int, int], np.ndarray]] = None\n        self.custom_goal_fn: Optional[Callable[[np.ndarray], bool]] = None\n        self.last_backtest_score: float = 0.0\n\n    def register_custom_simulator(\n        self,\n        step_fn: Callable[[np.ndarray, int, int, int], np.ndarray],\n        goal_fn: Optional[Callable[[np.ndarray], bool]] = None,\n    ) -> None:\n        self.custom_step_fn = step_fn\n        self.custom_goal_fn = goal_fn\n\n    def simulate_step(\n        self, grid: np.ndarray, action_id: int, x: int = 0, y: int = 0\n    ) -> np.ndarray:\n        if self.custom_step_fn is not None:\n            try:\n                out = self.custom_step_fn(grid.copy(), int(action_id), int(x), int(y))\n                if isinstance(out, np.ndarray) and out.shape == grid.shape:\n                    return np.asarray(out, dtype=np.int8)\n            except Exception:\n                pass\n        nxt = np.asarray(grid, dtype=np.int8).copy()\n        bg = self.processor.induce_background_color(nxt)\n        if int(action_id) == 6:\n            cx, cy = int(x), int(y)\n            if 0 <= cy < nxt.shape[0] and 0 <= cx < nxt.shape[1]:\n                c_curr = int(nxt[cy, cx])\n                if c_curr in self.hypotheses.click_color_transitions:\n                    c_next = self.hypotheses.click_color_transitions[c_curr]\n                    ents = self.processor.extract_entities(nxt, bg_color=bg, apply_hud_mask=False)\n                    for ent in ents:\n                        if (cy, cx) in ent.pixels:\n                            for pr, pc in ent.pixels:\n                                nxt[pr, pc] = c_next\n                            return nxt\n                    nxt[cy, cx] = c_next\n            return nxt\n        if int(action_id) in self.hypotheses.displacement_vectors:\n            dr, dc = self.hypotheses.displacement_vectors[int(action_id)]\n            ents = self.processor.extract_entities(nxt, bg_color=bg)\n            av_pos = self.hypotheses.locate_avatar(ents)\n            if av_pos is not None:\n                av_ents = [\n                    e\n                    for e in ents\n                    if (int(round(e.centroid[0])), int(round(e.centroid[1]))) == av_pos\n                ]\n                if av_ents:\n                    av_ent = min(av_ents, key=lambda e: e.mass)\n                    new_pixels = [(r + dr, c + dc) for r, c in av_ent.pixels]\n                    if all(0 <= r < nxt.shape[0] and 0 <= c < nxt.shape[1] for r, c in new_pixels):\n                        for r, c in av_ent.pixels:\n                            nxt[r, c] = bg\n                        for r, c in new_pixels:\n                            nxt[r, c] = av_ent.color\n        return nxt\n\n    def run_backtest(self, history: List[TransitionRecord]) -> float:\n        \"\"\"Replays the world model across recorded transitions to compute retrodictive accuracy.\"\"\"\n        valid_records = [\n            r for r in history if r.pre_grid is not None and r.post_grid is not None and r.delta_omega > 0\n        ]\n        if not valid_records:\n            return 0.0\n        matches = 0\n        for rec in valid_records[-25:]:\n            pred = self.simulate_step(rec.pre_grid, rec.action_id, rec.x, rec.y)\n            h_pred = compute_arena_hash(pred, self.processor.hud_masker)\n            if h_pred == rec.post_hash:\n                matches += 1\n        self.last_backtest_score = float(matches) / float(len(valid_records[-25:]))\n        return self.last_backtest_score\n\n    def infer_obstacle_cells(\n        self, grid: np.ndarray, entities: List[Entity], known_walls: Set[Tuple[int, int]]\n    ) -> Set[Tuple[int, int]]:\n        \"\"\"\n        Combines empirically blocked cells with pixels belonging to inferred wall colors\n        or elongated barrier structures (aspect ratio >= 3:1 or mass >= 4).\n        \"\"\"\n        obs_cells: Set[Tuple[int, int]] = set(known_walls)\n        for e in entities:\n            r0, c0, r1, c1 = e.bbox\n            h_box = r1 - r0 + 1\n            w_box = c1 - c0 + 1\n            is_elongated_barrier = e.mass >= 4 and (\n                h_box >= 3 * w_box or w_box >= 3 * h_box or e.mass >= 6\n            )\n            if (\n                e.color in self.hypotheses.wall_colors\n                or e.color in self.hypotheses.hazard_colors\n                or is_elongated_barrier\n            ):\n                for pr, pc in e.pixels:\n                    obs_cells.add((pr, pc))\n        return obs_cells\n\n    def plan_interactive_route(\n        self,\n        grid: np.ndarray,\n        entities: List[Entity],\n        av_pos: Tuple[int, int],\n        walls: Set[Tuple[int, int]],\n        visited_centers: Set[Tuple[int, int]],\n        enable_sokoban: bool = True,\n    ) -> Optional[List[int]]:\n        \"\"\"\n        Plans shortest A* route to unvisited interactive objects (switches, keys, goals, portals)\n        or box-pushing alignments using inferred displacement vectors.\n        \"\"\"\n        if not self.hypotheses.displacement_vectors:\n            return None\n        av_colors = self.hypotheses.avatar_colors\n        effective_walls = self.infer_obstacle_cells(grid, entities, walls)\n\n        candidates: List[Entity] = []\n        for e in entities:\n            rc = (int(round(e.centroid[0])), int(round(e.centroid[1])))\n            if rc == av_pos:\n                continue\n            if rc in effective_walls:\n                continue\n            if e.color in self.hypotheses.wall_colors or e.color in self.hypotheses.hazard_colors:\n                continue\n            if e.color in av_colors and e.mass <= 4:\n                continue\n            if rc in visited_centers and e.color not in self.hypotheses.pushable_colors:\n                continue\n            if 1 <= e.mass <= 36:\n                candidates.append(e)\n        if not candidates:\n            return None\n\n        if enable_sokoban and len(candidates) >= 2:\n            push_ents = [e for e in candidates if e.color in self.hypotheses.pushable_colors]\n            if not push_ents and len(candidates) == 2:\n                c_sorted = sorted(\n                    candidates,\n                    key=lambda e: abs(int(round(e.centroid[0])) - av_pos[0])\n                    + abs(int(round(e.centroid[1])) - av_pos[1]),\n                )\n                if c_sorted[0].color != c_sorted[1].color:\n                    push_ents = [c_sorted[0]]\n            target_ents = [e for e in candidates if e not in push_ents]\n            if push_ents and target_ents:\n                box = push_ents[0]\n                goal = target_ents[0]\n                br, bc = int(round(box.centroid[0])), int(round(box.centroid[1]))\n                gr, gc = int(round(goal.centroid[0])), int(round(goal.centroid[1]))\n                for act_id, (dr, dc) in self.hypotheses.displacement_vectors.items():\n                    if (dr, dc) == (0, 0):\n                        continue\n                    nbr, nbc = br + dr, bc + dc\n                    push_stand = (br - dr, bc - dc)\n                    old_d = abs(br - gr) + abs(bc - gc)\n                    nd = abs(nbr - gr) + abs(nbc - gc)\n                    if (\n                        nd < old_d\n                        and 0 <= push_stand[0] < grid.shape[0]\n                        and 0 <= push_stand[1] < grid.shape[1]\n                        and push_stand not in effective_walls\n                        and (nbr, nbc) not in effective_walls\n                    ):\n                        n_pushes = 1\n                        cr_b, cc_b = nbr, nbc\n                        while (cr_b, cc_b) != (gr, gc):\n                            nx_b, ny_b = cr_b + dr, cc_b + dc\n                            if abs(nx_b - gr) + abs(ny_b - gc) < abs(cr_b - gr) + abs(cc_b - gc):\n                                n_pushes += 1\n                                cr_b, cc_b = nx_b, ny_b\n                            else:\n                                break\n                        if av_pos == push_stand:\n                            return [int(act_id)] * n_pushes\n                        path_to_stand = astar_grid(\n                            av_pos,\n                            push_stand,\n                            effective_walls | {(br, bc)},\n                            self.hypotheses.displacement_vectors,\n                            grid.shape,\n                            portals=self.hypotheses.portal_map,\n                        )\n                        if path_to_stand is not None:\n                            return path_to_stand + ([int(act_id)] * n_pushes)\n\n        def _target_key(ent: Entity) -> Tuple[float, float]:\n            rc = (int(round(ent.centroid[0])), int(round(ent.centroid[1])))\n            dist = abs(rc[0] - av_pos[0]) + abs(rc[1] - av_pos[1])\n            prio = self.hypotheses.bayesian_tracker.exploration_priority(ent.shape_signature, 1)\n            if ent.color in self.hypotheses.goal_colors or ent.color in self.hypotheses.switch_colors:\n                prio += 0.5\n            return (-round(prio, 2), float(dist))\n\n        candidates.sort(key=_target_key)\n\n        for ent in candidates[:6]:\n            target_rc = (int(round(ent.centroid[0])), int(round(ent.centroid[1])))\n            path = astar_grid(\n                av_pos,\n                target_rc,\n                effective_walls,\n                self.hypotheses.displacement_vectors,\n                grid.shape,\n                portals=self.hypotheses.portal_map,\n            )\n            if path:\n                return path\n        return None\n\n    def plan_click_pattern_match(\n        self,\n        grid: np.ndarray,\n        entities: List[Entity],\n        arena_hash: int,\n        ledger: Any,\n        level_idx: int = 0,\n    ) -> Optional[Tuple[int, Dict[str, int], Optional[Tuple[int, int, int, int]]]]:\n        \"\"\"\n        Detects reference vs modifiable tile patterns (e.g. symmetry / color-cycle puzzles like ft09)\n        and selects the exact mismatched tile to click via ACTION6.\n        \"\"\"\n        if len(entities) < 2:\n            return None\n        by_shape: Dict[Tuple[int, int, int], List[Entity]] = {}\n        for e in entities:\n            r0, c0, r1, c1 = e.bbox\n            key = (r1 - r0 + 1, c1 - c0 + 1, e.mass)\n            by_shape.setdefault(key, []).append(e)\n\n        for shape_key, group in sorted(by_shape.items(), key=lambda kv: -kv[0][2]):\n            if len(group) < 2:\n                continue\n            colors = [e.color for e in group]\n            unique_colors, counts = np.unique(colors, return_counts=True)\n            if len(unique_colors) >= 2:\n                minority_color = int(unique_colors[int(np.argmin(counts))])\n                for ent in group:\n                    if ent.color == minority_color:\n                        cy = int(round(ent.centroid[0]))\n                        cx = int(round(ent.centroid[1]))\n                        if (cy, cx) not in ent.pixels and ent.pixels:\n                            cy, cx = ent.pixels[len(ent.pixels) // 2]\n                        sig = ent.shape_signature\n                        if not ledger.is_dead(arena_hash, 6, cx, cy, level_idx, obj_sig=sig):\n                            return 6, {\"x\": int(cx), \"y\": int(cy)}, sig\n        return None\n",
  "project_override/llm/__init__.py": "from project_override.llm.local_runtime import LocalLLMRuntime, parse_llm_response\nfrom project_override.llm.repl import SandboxedREPL\n\n__all__ = [\"LocalLLMRuntime\", \"parse_llm_response\", \"SandboxedREPL\"]\n",
  "project_override/llm/local_runtime.py": "import json\nimport logging\nimport os\nfrom pathlib import Path\nimport re\nimport threading\nimport time\nimport urllib.request\nfrom typing import Any, Dict, List, Optional\nfrom project_override.config import AgentConfig\n\nlogger = logging.getLogger(__name__)\n\ndef parse_llm_response(text: str) -> Optional[Dict[str, Any]]:\n    \"\"\"\n    Robust parser supporting fenced ```python ... ``` blocks (Duck REPL style)\n    and balanced JSON objects with self-repair.\n    \"\"\"\n    if not text:\n        return None\n    py_match = re.search(r\"```python\\s*(.*?)```\", text, re.DOTALL | re.IGNORECASE)\n    if py_match:\n        code_body = py_match.group(1).strip()\n        if code_body:\n            thought_prefix = text[: py_match.start()].strip()\n            return {\n                \"subgoal\": \"CODEACT\",\n                \"code\": code_body,\n                \"thought\": thought_prefix[:240],\n            }\n    dec = json.JSONDecoder()\n    for idx, ch in enumerate(text):\n        if ch == \"{\":\n            try:\n                obj, _ = dec.raw_decode(text[idx:])\n                if isinstance(obj, dict):\n                    return obj\n            except Exception:\n                continue\n    return None\n\nclass LocalLLMRuntime:\n    \"\"\"\n    Offline open-weight LLM loader supporting local SGLang/vLLM HTTP endpoints\n    (for Intel/Qwen3.8-Flash-Next-W4A16-AutoRound + albucino speculative drafter MTP=3)\n    and local HuggingFace checkpoints, with hard latency/token budgets and non-LLM fallback.\n    \"\"\"\n    _INSTANCE: Optional[\"LocalLLMRuntime\"] = None\n    _LOCK = threading.Lock()\n\n    def __init__(self, config: Optional[AgentConfig] = None):\n        self.config = config or AgentConfig()\n        self.http_endpoint: Optional[str] = None\n        self.http_model_name: str = \"default\"\n        self.model_path: Optional[str] = self._resolve_model_path()\n        self.drafter_path: Optional[str] = self._resolve_drafter_path()\n        self.model: Any = None\n        self.tokenizer: Any = None\n        self.is_loaded: bool = False\n        self.load_attempted: bool = False\n        self.total_calls: int = 0\n        self.fallback_count: int = 0\n\n    @classmethod\n    def get_instance(cls, config: Optional[AgentConfig] = None) -> \"LocalLLMRuntime\":\n        if cls._INSTANCE is None:\n            with cls._LOCK:\n                if cls._INSTANCE is None:\n                    cls._INSTANCE = cls(config=config)\n        return cls._INSTANCE\n\n    def _resolve_model_path(self) -> Optional[str]:\n        env_p = os.environ.get(\"ARC_LOCAL_MODEL_PATH\")\n        if env_p and Path(env_p).exists():\n            return env_p\n        for p in self.config.candidate_model_paths:\n            if Path(p).exists():\n                return p\n        kaggle_in = Path(\"/kaggle/input\")\n        if kaggle_in.exists():\n            for cfg in kaggle_in.rglob(\"config.json\"):\n                low = str(cfg).lower()\n                if \"drafter\" in low:\n                    continue\n                if any(k in low for k in (\"qwen\", \"gemma\", \"coder\", \"flash\", \"autoround\")):\n                    return str(cfg.parent)\n        return None\n\n    def _resolve_drafter_path(self) -> Optional[str]:\n        env_d = os.environ.get(\"ARC_LOCAL_DRAFTER_PATH\")\n        if env_d and Path(env_d).exists():\n            return env_d\n        for p in self.config.candidate_drafter_paths:\n            if Path(p).exists():\n                return p\n        kaggle_in = Path(\"/kaggle/input\")\n        if kaggle_in.exists():\n            for cfg in kaggle_in.rglob(\"config.json\"):\n                if \"drafter\" in str(cfg).lower():\n                    return str(cfg.parent)\n        return None\n\n    def _probe_local_http_server(self) -> bool:\n        for base_url in self.config.vllm_base_urls:\n            if not base_url.startswith((\"http://127.0.0.1\", \"http://localhost\")):\n                continue\n            try:\n                req = urllib.request.Request(f\"{base_url}/models\", method=\"GET\")\n                with urllib.request.urlopen(req, timeout=1.5) as resp:\n                    data = json.loads(resp.read().decode(\"utf-8\"))\n                    models = data.get(\"data\", [])\n                    if models:\n                        self.http_model_name = models[0].get(\"id\", \"default\")\n                    self.http_endpoint = f\"{base_url}/chat/completions\"\n                    self.is_loaded = True\n                    return True\n            except Exception:\n                continue\n        return False\n\n    def ensure_loaded(self) -> bool:\n        if not self.config.enable_llm:\n            return False\n        if self.is_loaded:\n            return True\n        if self.load_attempted:\n            return False\n        with self._LOCK:\n            if self.is_loaded:\n                return True\n            self.load_attempted = True\n            if self._probe_local_http_server():\n                logger.info(f\"Connected to local SGLang/vLLM server at {self.http_endpoint}\")\n                return True\n            if self.model_path is None:\n                return False\n            try:\n                import torch\n                if not torch.cuda.is_available():\n                    return False\n                from transformers import AutoModelForCausalLM, AutoTokenizer\n                self.tokenizer = AutoTokenizer.from_pretrained(\n                    self.model_path, trust_remote_code=True, local_files_only=True\n                )\n                max_mem = {i: \"14GiB\" for i in range(torch.cuda.device_count())}\n                max_mem[\"cpu\"] = \"24GiB\"\n                self.model = AutoModelForCausalLM.from_pretrained(\n                    self.model_path,\n                    device_map=\"auto\",\n                    max_memory=max_mem,\n                    torch_dtype=torch.float16,\n                    trust_remote_code=True,\n                    local_files_only=True,\n                )\n                self.model.eval()\n                self.is_loaded = True\n                return True\n            except Exception as exc:\n                logger.warning(f\"LocalLLMRuntime load failed; using World-Model/BFS fallback: {exc}\")\n                self.is_loaded = False\n                return False\n\n    def generate_text(\n        self,\n        prompt: str,\n        max_new_tokens: Optional[int] = None,\n        timeout_s: Optional[float] = None,\n    ) -> Optional[str]:\n        self.total_calls += 1\n        max_tok = max_new_tokens or self.config.llm_max_new_tokens\n        limit_s = timeout_s or self.config.llm_timeout_s\n        if not self.ensure_loaded():\n            self.fallback_count += 1\n            return None\n        if self.http_endpoint is not None:\n            try:\n                payload = json.dumps(\n                    {\n                        \"model\": self.http_model_name,\n                        \"messages\": [{\"role\": \"user\", \"content\": prompt}],\n                        \"max_tokens\": max_tok,\n                        \"temperature\": 0.1,\n                        \"chat_template_kwargs\": {\"enable_thinking\": False},\n                    }\n                ).encode(\"utf-8\")\n                req = urllib.request.Request(\n                    self.http_endpoint,\n                    data=payload,\n                    headers={\"Content-Type\": \"application/json\"},\n                    method=\"POST\",\n                )\n                with urllib.request.urlopen(req, timeout=limit_s) as resp:\n                    body = json.loads(resp.read().decode(\"utf-8\"))\n                    return body[\"choices\"][0][\"message\"][\"content\"].strip()\n            except Exception as exc:\n                logger.debug(f\"HTTP LLM inference error: {exc}\")\n                self.fallback_count += 1\n                return None\n        try:\n            import torch\n            t0 = time.perf_counter()\n            messages = [{\"role\": \"user\", \"content\": prompt}]\n            text = (\n                self.tokenizer.apply_chat_template(\n                    messages, tokenize=False, add_generation_prompt=True\n                )\n                if hasattr(self.tokenizer, \"apply_chat_template\")\n                else prompt\n            )\n            inputs = self.tokenizer(text, return_tensors=\"pt\").to(self.model.device)\n            if (time.perf_counter() - t0) > limit_s:\n                self.fallback_count += 1\n                return None\n            with torch.no_grad():\n                out = self.model.generate(\n                    **inputs,\n                    max_new_tokens=max_tok,\n                    do_sample=False,\n                    use_cache=True,\n                    pad_token_id=self.tokenizer.eos_token_id,\n                )\n            gen_ids = out[0][inputs[\"input_ids\"].shape[1] :]\n            return self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()\n        except Exception as exc:\n            logger.debug(f\"Transformers LLM inference error: {exc}\")\n            self.fallback_count += 1\n            return None\n\n    def infer_decision(self, prompt: str) -> Optional[Dict[str, Any]]:\n        raw = self.generate_text(prompt)\n        if not raw:\n            return None\n        return parse_llm_response(raw)\n",
  "project_override/llm/repl.py": "import ast\nfrom typing import Any, Callable, Dict, List, Optional, Tuple\nimport numpy as np\nfrom project_override.perception.observation_processor import Entity\n\nclass SandboxedREPL:\n    \"\"\"\n    Tufa Labs Duck-style + Schema/PRO-LONG Python REPL with AST safety validation.\n    Allows the model to inspect `grid`, `prev_grid`, `objects`, `avail_actions`,\n    query recorded history via `query_log(action_id=None, min_delta=0)`,\n    register an executable simulator via `register_world_model(step_fn, goal_fn=None)`,\n    update persistent `notes`, and emit actions via `act(id, x=0, y=0)` or `plan`.\n    \"\"\"\n    FORBIDDEN_NODES = (\n        ast.Import,\n        ast.ImportFrom,\n        ast.Global,\n        ast.Nonlocal,\n    )\n    FORBIDDEN_NAMES = {\n        \"eval\",\n        \"exec\",\n        \"compile\",\n        \"open\",\n        \"__import__\",\n        \"getattr\",\n        \"setattr\",\n        \"delattr\",\n        \"globals\",\n        \"locals\",\n        \"input\",\n    }\n\n    def __init__(self, max_plan_len: int = 14):\n        self.max_plan_len = max_plan_len\n\n    def validate_ast(self, code_str: str) -> bool:\n        try:\n            tree = ast.parse(code_str)\n        except SyntaxError:\n            return False\n        for node in ast.walk(tree):\n            if isinstance(node, self.FORBIDDEN_NODES):\n                return False\n            if isinstance(node, ast.Name) and node.id in self.FORBIDDEN_NAMES:\n                return False\n            if isinstance(node, ast.Attribute) and node.attr.startswith(\"__\"):\n                return False\n        return True\n\n    def execute_codeact(\n        self,\n        code_str: str,\n        grid: np.ndarray,\n        prev_grid: Optional[np.ndarray],\n        av_pos: Optional[Tuple[int, int]],\n        bg_color: int,\n        entities: List[Entity],\n        vectors: Dict[int, Tuple[int, int]],\n        avail_actions: List[int],\n        notes: Dict[str, str],\n        query_log_fn: Optional[Callable[..., List[Dict[str, Any]]]] = None,\n        register_wm_fn: Optional[Callable[..., None]] = None,\n    ) -> List[Tuple[int, Dict[str, int]]]:\n        if not self.validate_ast(code_str):\n            return []\n        emitted: List[Tuple[int, Dict[str, int]]] = []\n\n        def act(action_id: int, x: int = 0, y: int = 0) -> None:\n            aid = int(action_id)\n            if aid in avail_actions and len(emitted) < self.max_plan_len:\n                if aid == 6:\n                    emitted.append(\n                        (6, {\"x\": max(0, min(63, int(x))), \"y\": max(0, min(63, int(y)))})\n                    )\n                else:\n                    emitted.append((aid, {}))\n\n        def note(key: str, val: Any) -> None:\n            notes[str(key)[:40]] = str(val)[:160]\n\n        def query_log(action_id: Optional[int] = None, min_delta: int = 0) -> List[Dict[str, Any]]:\n            if query_log_fn is not None:\n                return query_log_fn(action_id=action_id, min_delta=min_delta)\n            return []\n\n        def register_world_model(step_fn: Any, goal_fn: Any = None) -> None:\n            if register_wm_fn is not None and callable(step_fn):\n                register_wm_fn(step_fn, goal_fn if callable(goal_fn) else None)\n\n        safe_builtins = {\n            \"abs\": abs,\n            \"min\": min,\n            \"max\": max,\n            \"sum\": sum,\n            \"len\": len,\n            \"range\": range,\n            \"enumerate\": enumerate,\n            \"zip\": zip,\n            \"sorted\": sorted,\n            \"round\": round,\n            \"int\": int,\n            \"float\": float,\n            \"str\": str,\n            \"bool\": bool,\n            \"list\": list,\n            \"dict\": dict,\n            \"set\": set,\n            \"tuple\": tuple,\n            \"any\": any,\n            \"all\": all,\n        }\n        obj_dicts = [\n            {\n                \"id\": e.id,\n                \"color\": e.color,\n                \"mass\": e.mass,\n                \"r\": int(round(e.centroid[0])),\n                \"c\": int(round(e.centroid[1])),\n                \"bbox\": e.bbox,\n            }\n            for e in entities\n        ]\n        local_env: Dict[str, Any] = {\n            \"__builtins__\": safe_builtins,\n            \"np\": np,\n            \"grid\": grid.copy(),\n            \"prev_grid\": prev_grid.copy() if prev_grid is not None else grid.copy(),\n            \"av_pos\": av_pos,\n            \"bg_color\": bg_color,\n            \"objects\": obj_dicts,\n            \"vectors\": dict(vectors),\n            \"avail_actions\": list(avail_actions),\n            \"notes\": notes,\n            \"act\": act,\n            \"note\": note,\n            \"query_log\": query_log,\n            \"register_world_model\": register_world_model,\n            \"plan\": [],\n        }\n        try:\n            exec(compile(code_str, \"<repl>\", \"exec\"), local_env, local_env)\n        except Exception:\n            return emitted[: self.max_plan_len]\n        if not emitted and isinstance(local_env.get(\"plan\"), list):\n            for item in local_env[\"plan\"][: self.max_plan_len]:\n                if isinstance(item, int) and item in avail_actions:\n                    emitted.append((item, {}))\n                elif isinstance(item, tuple) and len(item) == 2 and int(item[0]) in avail_actions:\n                    emitted.append((int(item[0]), dict(item[1])))\n                elif (\n                    isinstance(item, tuple)\n                    and len(item) == 3\n                    and int(item[0]) == 6\n                    and 6 in avail_actions\n                ):\n                    emitted.append((6, {\"x\": int(item[1]), \"y\": int(item[2])}))\n        return emitted[: self.max_plan_len]\n",
  "project_override/execution/__init__.py": "from project_override.execution.safe_executor import (\n    DPrimeComputeScheduler,\n    SafeGameExecutor,\n    TrajectorySupervisor,\n)\n\n__all__ = [\"DPrimeComputeScheduler\", \"SafeGameExecutor\", \"TrajectorySupervisor\"]\n",
  "project_override/execution/safe_executor.py": "import logging\nimport time\nfrom typing import Any, Callable, Dict, Optional, Set, Tuple\nfrom project_override.config import AgentConfig\n\nlogger = logging.getLogger(__name__)\n\nclass TrajectorySupervisor:\n    \"\"\"\n    NVIDIA AVO Trajectory Supervisor (arXiv:2603.24517) + Retrodict Tiered Escalation:\n    Monitors the agent's search trajectory for stagnation or unproductive loops,\n    redirects exploration across escalation tiers, and enforces strict RESET discipline\n    (never issuing two RESETs in a row so the game never resets back to Level 1).\n    \"\"\"\n    def __init__(self, enabled: bool = True):\n        self.enabled = enabled\n        self.seen_hashes_in_level: Set[int] = set()\n        self.steps_since_novel_state: int = 0\n        self.consecutive_resets: int = 0\n        self.escalation_tier: int = 0\n        self.interventions: int = 0\n\n    def reset_level(self) -> None:\n        self.seen_hashes_in_level.clear()\n        self.steps_since_novel_state = 0\n        self.escalation_tier = 0\n\n    def observe_state(self, state_hash: int, action_id: Optional[int]) -> None:\n        if action_id == 0:\n            self.consecutive_resets += 1\n        elif action_id is not None:\n            self.consecutive_resets = 0\n        if state_hash not in self.seen_hashes_in_level:\n            self.seen_hashes_in_level.add(state_hash)\n            self.steps_since_novel_state = 0\n            if self.escalation_tier > 0:\n                self.escalation_tier = max(0, self.escalation_tier - 1)\n        else:\n            self.steps_since_novel_state += 1\n            if self.enabled:\n                if self.steps_since_novel_state >= 6:\n                    self.escalation_tier = 2\n                    self.interventions += 1\n                elif self.steps_since_novel_state >= 3:\n                    self.escalation_tier = max(self.escalation_tier, 1)\n                    self.interventions += 1\n\n    def can_issue_reset(self) -> bool:\n        \"\"\"Enforces Retrodict RESET discipline: never issue two RESETs in a row.\"\"\"\n        return self.consecutive_resets == 0\n\nclass DPrimeComputeScheduler:\n    \"\"\"\n    Daniel Franzen Milestone 2 / D-Prime style dynamic compute scheduler:\n    Tracks wall-clock budget across all games and allocates extra steps/time\n    to games where levels are actively being cleared or state novelty is high.\n    \"\"\"\n    def __init__(self, config: Optional[AgentConfig] = None):\n        self.config = config or AgentConfig()\n        self.start_time = time.perf_counter()\n        self.games_started = 0\n        self.games_completed = 0\n        self.total_levels_cleared = 0\n\n    def elapsed_total_s(self) -> float:\n        return time.perf_counter() - self.start_time\n\n    def remaining_total_s(self) -> float:\n        return max(10.0, self.config.total_kaggle_budget_s - self.elapsed_total_s())\n\n    def allocate_game_budget(\n        self, levels_cleared_in_game: int, current_novelty_rate: float = 1.0\n    ) -> Tuple[int, float]:\n        rem_games = max(1, self.config.expected_num_games - self.games_completed)\n        base_time_s = self.remaining_total_s() / float(rem_games)\n        base_steps = self.config.max_actions_per_game\n        if levels_cleared_in_game >= 2:\n            multiplier = 1.75\n        elif levels_cleared_in_game == 1 or current_novelty_rate > 0.5:\n            multiplier = 1.35\n        else:\n            multiplier = 1.0\n        max_steps = int(min(700, base_steps * multiplier))\n        max_time_s = float(min(900.0, base_time_s * multiplier))\n        return max_steps, max_time_s\n\nclass SafeGameExecutor:\n    \"\"\"Isolates per-action and per-game exceptions so no single error can halt the run.\"\"\"\n    def __init__(self, scheduler: Optional[DPrimeComputeScheduler] = None):\n        self.scheduler = scheduler or DPrimeComputeScheduler()\n\n    def safe_step(\n        self,\n        step_fn: Callable[[], Tuple[int, Dict[str, Any]]],\n        default_action: int = 1,\n    ) -> Tuple[int, Dict[str, Any]]:\n        try:\n            act_id, payload = step_fn()\n            return int(act_id), dict(payload or {})\n        except Exception as exc:\n            logger.warning(f\"SafeGameExecutor caught action exception: {exc}\")\n            return int(default_action), {}\n",
  "project_override/orchestration/__init__.py": "from project_override.orchestration.general_agent import GeneralAgent\n\n__all__ = [\"GeneralAgent\"]\n",
  "project_override/orchestration/general_agent.py": "from collections import deque\nimport logging\nfrom typing import Any, Dict, List, Optional, Set, Tuple\nimport numpy as np\n\nfrom project_override.config import AgentConfig\nfrom project_override.execution.safe_executor import (\n    DPrimeComputeScheduler,\n    SafeGameExecutor,\n    TrajectorySupervisor,\n)\nfrom project_override.llm.local_runtime import LocalLLMRuntime\nfrom project_override.llm.repl import SandboxedREPL\nfrom project_override.memory.action_ledger import ActionLedger, compute_arena_hash\nfrom project_override.memory.hypothesis_store import HypothesisStore\nfrom project_override.perception.observation_processor import ObservationProcessor\nfrom project_override.planning.explorer import SmartClickExplorer, bfs_frontier_explore\nfrom project_override.planning.planner import EmpiricalWorldModel, MacroActionQueue, astar_grid\n\nlogger = logging.getLogger(__name__)\n\nclass GeneralAgent:\n    \"\"\"\n    General, observation-driven ARC-AGI-3 agent combining:\n    - Live Sandboxed Python REPL + PRO-LONG log query (Tufa Duck / PRO-LONG arXiv:2607.20064)\n    - Executable World-Model Induction, Retrodiction & Goal-Directed BFS (Tycho / Schema / Kepler)\n    - OPINE-World Lifted Bayesian Exploration Guidance S_B (arXiv:2607.01531)\n    - NVIDIA AVO Trajectory Supervisor & Tiered Escalation (arXiv:2603.24517 / Retrodict)\n    - Interior-board no-op guards & ACTION7 Undo backtracking (Franzen M2)\n    - Dead-signature & object-class click pruning with No-Total-Coverage safety (Reki M1 / Kepler)\n    - S4 Reasoning Replay & Two-Part Playbook Memory (arXiv:2610.26632 / Retrodict)\n    \"\"\"\n    def __init__(self, config: Optional[AgentConfig] = None):\n        self.config = config or AgentConfig()\n        self.processor = ObservationProcessor(enable_hud_masking=self.config.enable_hud_masking)\n        self.ledger = ActionLedger(enable_dead_signatures=self.config.enable_dead_signatures)\n        self.hypotheses = HypothesisStore()\n        self.world_model = EmpiricalWorldModel(self.processor, self.hypotheses)\n        self.macro_queue = MacroActionQueue()\n        self.click_explorer = SmartClickExplorer(self.processor)\n        self.repl = SandboxedREPL(max_plan_len=self.config.max_repl_plan_len)\n        self.scheduler = DPrimeComputeScheduler(self.config)\n        self.supervisor = TrajectorySupervisor(enabled=self.config.enable_avo_supervisor)\n        self.executor = SafeGameExecutor(self.scheduler)\n        self.current_level: int = 0\n        self.step_in_game: int = 0\n        self.step_in_level: int = 0\n        self.prev_grid: Optional[np.ndarray] = None\n        self.prev_hash: Optional[int] = None\n        self.last_action_id: Optional[int] = None\n        self.last_click_xy: Tuple[int, int] = (0, 0)\n        self.last_clicked_sig: Optional[Tuple[int, int, int, int]] = None\n        self.last_delta_omega: Optional[int] = None\n        self.visited_cells: Set[Tuple[int, int]] = set()\n        self.wall_cells: Set[Tuple[int, int]] = set()\n        self.rolling_turns: deque = deque(maxlen=self.config.max_rolling_turns)\n        self.last_llm_step: int = -999\n\n    def reset_game(self) -> None:\n        self.current_level = 0\n        self.step_in_game = 0\n        self.step_in_level = 0\n        self.last_llm_step = -999\n        self.prev_grid = None\n        self.prev_hash = None\n        self.last_action_id = None\n        self.last_click_xy = (0, 0)\n        self.last_clicked_sig = None\n        self.last_delta_omega = None\n        self.visited_cells.clear()\n        self.wall_cells.clear()\n        self.rolling_turns.clear()\n        self.ledger.reset_level(clear_deaths=True)\n        self.hypotheses = HypothesisStore()\n        self.world_model = EmpiricalWorldModel(self.processor, self.hypotheses)\n        self.macro_queue.clear()\n        self.supervisor.reset_level()\n        self.processor.hud_masker.reset_level()\n\n    def _on_level_advance(self, new_level: int) -> None:\n        self.hypotheses.reset_for_new_level(cleared_previous=True, prev_level=self.current_level)\n        self.current_level = new_level\n        self.step_in_level = 0\n        self.last_llm_step = -999\n        self.prev_grid = None\n        self.prev_hash = None\n        self.last_action_id = None\n        self.last_delta_omega = None\n        self.visited_cells.clear()\n        self.wall_cells.clear()\n        self.ledger.reset_level(clear_deaths=False)\n        self.macro_queue.clear()\n        self.supervisor.reset_level()\n        self.processor.hud_masker.reset_level()\n\n    def build_repl_prompt(\n        self,\n        grid: np.ndarray,\n        avail_actions: List[int],\n        av_pos: Optional[Tuple[int, int]],\n        diff_str: str,\n    ) -> str:\n        obs_str = self.processor.encode_compact_observation(grid)\n        epistemic_str = self.hypotheses.format_epistemic_memory()\n        recent_str = \"\\n\".join(self.rolling_turns) if self.rolling_turns else \"None\"\n        untried = self.ledger.get_untried_discrete_actions(\n            compute_arena_hash(grid, self.processor.hud_masker),\n            avail_actions,\n            self.current_level,\n        )\n        return (\n            \"ROLE: Autonomous ARC-AGI-3 scientist & coding agent (Duck REPL + Retrodict + Schema).\\n\"\n            \"DIRECTIVES:\\n\"\n            \"1. RETRODICTION: Check hypotheses against recorded history via query_log() before acting.\\n\"\n            \"2. EXPLORE THEN COMMIT: Probe uncertain mechanics with single moves; batch predictable paths.\\n\"\n            \"3. HUD GUIDANCE: Ignore border strips that tick on every step (timers/step counters).\\n\"\n            \"4. RESET DISCIPLINE: Use ACTION7 (Undo) when stuck; never issue two RESETs in a row.\\n\"\n            f\"LEVEL: {self.current_level} | STEP_IN_LEVEL: {self.step_in_level} | ESCALATION_TIER: {self.supervisor.escalation_tier}\\n\"\n            f\"AVAIL_ACTIONS: {avail_actions} | UNTRIED_DISCRETE: {untried}\\n\"\n            f\"AVATAR_RC: {av_pos} | LAST_DIFF: {diff_str} | WM_BACKTEST_ACC: {self.world_model.last_backtest_score:.2f}\\n\"\n            f\"{epistemic_str}\\n\"\n            f\"RECENT_TURNS:\\n{recent_str}\\n\"\n            f\"CURRENT_OBSERVATION:\\n{obs_str}\\n\\n\"\n            \"INSTRUCTIONS:\\n\"\n            \"Respond with either a ```python ... ``` block calling act(action_id, x=0, y=0), note(k, v),\\n\"\n            \"or register_world_model(step_fn, goal_fn), OR a single JSON object:\\n\"\n            '{\"thought\": \"...\", \"subgoal\": \"ACTION\", \"action_id\": 1} or {\"thought\": \"...\", \"subgoal\": \"CLICK\", \"click_xy\": [x, y]}.'\n        )\n\n    def act(\n        self,\n        raw_obs: Any,\n        available_actions: Optional[List[int]] = None,\n        levels_completed: int = 0,\n        state_str: str = \"PLAYING\",\n    ) -> Tuple[int, Dict[str, Any]]:\n        return self.executor.safe_step(\n            lambda: self._act_internal(raw_obs, available_actions, levels_completed, state_str),\n            default_action=1,\n        )\n\n    def _act_internal(\n        self,\n        raw_obs: Any,\n        available_actions: Optional[List[int]] = None,\n        levels_completed: int = 0,\n        state_str: str = \"PLAYING\",\n    ) -> Tuple[int, Dict[str, Any]]:\n        if state_str in (\"NOT_PLAYED\", \"GAME_OVER\"):\n            if (\n                state_str == \"GAME_OVER\"\n                and self.prev_hash is not None\n                and self.last_action_id is not None\n            ):\n                self.ledger.record_death(\n                    self.prev_hash,\n                    self.last_action_id,\n                    self.last_click_xy[0],\n                    self.last_click_xy[1],\n                    self.current_level,\n                )\n            self.macro_queue.clear()\n            self.prev_grid = None\n            self.prev_hash = None\n            if self.supervisor.can_issue_reset():\n                self.supervisor.observe_state(0, 0)\n                return 0, {}\n            return 1, {}\n\n        if levels_completed > self.current_level:\n            self._on_level_advance(levels_completed)\n\n        anim_frames = self.processor.extract_animation_sequence(raw_obs)\n        arr = anim_frames[-1]\n        avail = sorted({int(a) for a in (available_actions or [1, 2, 3, 4, 5, 6]) if int(a) != 0})\n        if not avail:\n            avail = [1, 2, 3, 4]\n\n        self.step_in_game += 1\n        self.step_in_level += 1\n\n        bg_color = self.processor.induce_background_color(arr)\n        curr_entities = self.processor.extract_entities(arr, bg_color=bg_color)\n        diff_str = \"INITIAL_FRAME\"\n\n        if (\n            self.prev_grid is not None\n            and self.prev_hash is not None\n            and self.last_action_id is not None\n        ):\n            self.processor.hud_masker.observe_transition(self.prev_grid, arr)\n            curr_hash = compute_arena_hash(arr, self.processor.hud_masker)\n            delta_omega = self.processor.compute_delta_omega(self.prev_grid, arr)\n            self.last_delta_omega = delta_omega\n            diff_str = self.processor.summarize_transition(self.prev_grid, arr, anim_frames)\n            prev_entities = self.processor.extract_entities(self.prev_grid, bg_color=bg_color)\n            tracked = self.processor.track_entities(prev_entities, curr_entities)\n            self.hypotheses.observe_entity_motions(\n                self.last_action_id,\n                tracked,\n                pre_grid=self.prev_grid,\n                post_grid=arr,\n                click_xy=self.last_click_xy,\n            )\n            if self.last_clicked_sig is not None:\n                eff_tag = \"NO_OP\" if delta_omega == 0 else f\"DELTA_{min(20, delta_omega)}\"\n                self.hypotheses.bayesian_tracker.record_effect(\n                    self.last_clicked_sig, self.last_action_id, eff_tag\n                )\n            if self.last_action_id in (1, 2, 3, 4) and delta_omega >= 28:\n                self.wall_cells.clear()\n                self.macro_queue.clear()\n            self.ledger.record_transition(\n                pre_hash=self.prev_hash,\n                action_id=self.last_action_id,\n                post_hash=curr_hash,\n                delta_omega=delta_omega,\n                coord_x=self.last_click_xy[0],\n                coord_y=self.last_click_xy[1],\n                level_idx=self.current_level,\n                pre_grid=self.prev_grid,\n                post_grid=arr,\n                diff_summary=diff_str,\n                clicked_obj_sig=self.last_clicked_sig,\n                step_idx=self.step_in_level,\n            )\n            self.world_model.run_backtest(self.ledger.history)\n            self.rolling_turns.append(\n                f\"S{self.step_in_level - 1}: A{self.last_action_id}{self.last_click_xy if self.last_action_id == 6 else ''} -> {diff_str}\"\n            )\n        else:\n            curr_hash = compute_arena_hash(arr, self.processor.hud_masker)\n            self.ledger.state_visit_counts[curr_hash] = (\n                self.ledger.state_visit_counts.get(curr_hash, 0) + 1\n            )\n            self.last_delta_omega = None\n\n        self.supervisor.observe_state(curr_hash, self.last_action_id)\n\n        av_pos = self.hypotheses.locate_avatar(curr_entities)\n        if av_pos is not None:\n            self.visited_cells.add(av_pos)\n            self.hypotheses.interacted_object_centers.add(av_pos)\n            if (\n                self.last_delta_omega == 0\n                and self.last_action_id in self.hypotheses.displacement_vectors\n            ):\n                dr, dc = self.hypotheses.displacement_vectors[self.last_action_id]\n                blocked_rc = (av_pos[0] + dr, av_pos[1] + dc)\n                self.wall_cells.add(blocked_rc)\n                if 0 <= blocked_rc[0] < arr.shape[0] and 0 <= blocked_rc[1] < arr.shape[1]:\n                    w_color = int(arr[blocked_rc[0], blocked_rc[1]])\n                    if w_color != bg_color and w_color not in self.hypotheses.pushable_colors:\n                        self.hypotheses.wall_colors.add(w_color)\n\n        self.prev_grid = arr.copy()\n        self.prev_hash = curr_hash\n        self.last_clicked_sig = None\n\n        # 1. Check guarded macro queue (Franzen M2 interior-board change guard + expectation check)\n        queued = self.macro_queue.pop_next(\n            last_delta_omega=self.last_delta_omega,\n            guard_enabled=self.config.enable_action_guards,\n            current_hash=curr_hash,\n        )\n        if queued is not None:\n            q_act, q_data = queued\n            if q_act in avail and not self.ledger.is_dead(\n                curr_hash, q_act, q_data.get(\"x\", 0), q_data.get(\"y\", 0), self.current_level\n            ):\n                self.last_action_id = q_act\n                self.last_click_xy = (q_data.get(\"x\", 0), q_data.get(\"y\", 0))\n                return q_act, q_data\n\n        # 2. AVO Supervisor + ACTION7 (UNDO) loop & dead-end trap recovery\n        move_disc_here = [a for a in avail if a in (1, 2, 3, 4)]\n        all_moves_dead = bool(move_disc_here) and all(\n            self.ledger.is_dead(curr_hash, a, 0, 0, self.current_level) for a in move_disc_here\n        )\n        if (\n            self.config.enable_undo_backtrack\n            and 7 in avail\n            and (\n                all_moves_dead\n                or self.ledger.is_in_loop()\n                or (self.config.enable_avo_supervisor and self.supervisor.escalation_tier >= 1)\n            )\n            and self.last_action_id != 7\n            and not self.ledger.is_dead(curr_hash, 7, 0, 0, self.current_level)\n        ):\n            if av_pos is not None:\n                self.wall_cells.add(av_pos)\n            self.macro_queue.clear()\n            self.last_action_id = 7\n            self.last_click_xy = (0, 0)\n            return 7, {}\n\n        # 3. Pattern-Matching Click Solver when ACTION6 is the primary interaction or mismatches exist\n        if self.config.enable_pattern_click_solver and 6 in avail:\n            has_move_actions = any(a in avail for a in (1, 2, 3, 4))\n            if not has_move_actions:\n                pattern_click = self.world_model.plan_click_pattern_match(\n                    arr, curr_entities, curr_hash, self.ledger, self.current_level\n                )\n                if pattern_click is not None:\n                    act_id, data, obj_sig = pattern_click\n                    self.last_action_id = act_id\n                    self.last_click_xy = (data[\"x\"], data[\"y\"])\n                    self.last_clicked_sig = obj_sig\n                    return act_id, data\n\n        # 4. Probe untried movement actions (1..4) at initial state to rapidly induce movement vectors\n        move_discrete = [a for a in avail if a in (1, 2, 3, 4)]\n        if len(self.hypotheses.displacement_vectors) < len(move_discrete) and self.step_in_level <= 6:\n            untried_moves = [\n                a\n                for a in self.ledger.get_untried_discrete_actions(\n                    curr_hash, move_discrete, self.current_level\n                )\n                if a not in self.hypotheses.displacement_vectors\n            ]\n            if untried_moves:\n                chosen = untried_moves[0]\n                self.last_action_id = chosen\n                self.last_click_xy = (0, 0)\n                return chosen, {}\n\n        # 5. Executable World-Model Goal/Switch/Box-Push Route Planner (Schema / Tycho / OPINE-World)\n        # When a verified route to an interactive object is known, execute with zero latency\n        if (\n            self.config.enable_world_model_bfs\n            and av_pos is not None\n            and self.hypotheses.displacement_vectors\n            and self.supervisor.escalation_tier == 0\n        ):\n            wm_route = self.world_model.plan_interactive_route(\n                arr,\n                curr_entities,\n                av_pos,\n                self.wall_cells,\n                self.hypotheses.interacted_object_centers,\n                enable_sokoban=self.config.enable_sokoban_planner,\n            )\n            if wm_route:\n                first_act = int(wm_route[0])\n                if first_act in avail and not self.ledger.is_dead(\n                    curr_hash, first_act, 0, 0, self.current_level\n                ):\n                    if len(wm_route) > 1:\n                        self.macro_queue.push_plan(\n                            wm_route[1:],\n                            subagent_tag=\"WORLD_MODEL_BFS\",\n                            max_len=self.config.max_repl_plan_len,\n                        )\n                    self.last_action_id = first_act\n                    self.last_click_xy = (0, 0)\n                    return first_act, {}\n\n        # 6. Invoke Local Open-Weight LLM + Live Python REPL (Tufa Duck / Franzen M2 / Schema style)\n        # Queried when no direct route exists, on level entry, or upon stagnation/escalation\n        steps_since_llm = self.step_in_level - self.last_llm_step\n        should_call_llm = (\n            self.config.enable_llm\n            and (steps_since_llm >= self.config.llm_call_interval)\n            and (\n                self.step_in_level == 1\n                or self.supervisor.escalation_tier >= 1\n                or (self.step_in_level % self.config.llm_call_interval == 0)\n            )\n        )\n        if should_call_llm:\n            self.last_llm_step = self.step_in_level\n            llm = LocalLLMRuntime.get_instance(self.config)\n            if llm.ensure_loaded():\n                prompt = self.build_repl_prompt(arr, avail, av_pos, diff_str)\n                decision = llm.infer_decision(prompt)\n                if decision:\n                    if isinstance(decision.get(\"thought\"), str):\n                        self.hypotheses.add_reasoning_trace(\n                            self.step_in_level, self.current_level, decision[\"thought\"]\n                        )\n                    subgoal = decision.get(\"subgoal\", \"\")\n                    if subgoal == \"CODEACT\" and isinstance(decision.get(\"code\"), str):\n                        repl_plan = self.repl.execute_codeact(\n                            code_str=decision[\"code\"],\n                            grid=arr,\n                            prev_grid=self.prev_grid,\n                            av_pos=av_pos,\n                            bg_color=bg_color,\n                            entities=curr_entities,\n                            vectors=self.hypotheses.displacement_vectors,\n                            avail_actions=avail,\n                            notes=self.hypotheses.persistent_notes,\n                            query_log_fn=self.ledger.query_history,\n                            register_wm_fn=self.world_model.register_custom_simulator,\n                        )\n                        valid_plan = [\n                            (aid, pdata)\n                            for aid, pdata in repl_plan\n                            if aid in avail\n                            and not self.ledger.is_dead(\n                                curr_hash,\n                                aid,\n                                pdata.get(\"x\", 0),\n                                pdata.get(\"y\", 0),\n                                self.current_level,\n                            )\n                        ]\n                        if valid_plan:\n                            first_aid, first_data = valid_plan[0]\n                            if len(valid_plan) > 1:\n                                self.macro_queue.push_plan(\n                                    valid_plan[1:],\n                                    subagent_tag=\"DUCK_REPL\",\n                                    max_len=self.config.max_repl_plan_len,\n                                )\n                            self.last_action_id = first_aid\n                            self.last_click_xy = (\n                                first_data.get(\"x\", 0),\n                                first_data.get(\"y\", 0),\n                            )\n                            return first_aid, first_data\n                    elif (\n                        subgoal == \"NAVIGATE\"\n                        and isinstance(decision.get(\"target_rc\"), list)\n                        and av_pos\n                    ):\n                        tr, tc = int(decision[\"target_rc\"][0]), int(decision[\"target_rc\"][1])\n                        nav_plan = astar_grid(\n                            av_pos,\n                            (tr, tc),\n                            self.wall_cells,\n                            self.hypotheses.displacement_vectors,\n                            arr.shape,\n                            portals=self.hypotheses.portal_map,\n                        )\n                        if nav_plan:\n                            first_aid = int(nav_plan[0])\n                            if not self.ledger.is_dead(\n                                curr_hash, first_aid, 0, 0, self.current_level\n                            ):\n                                self.macro_queue.push_plan(nav_plan[1:], subagent_tag=\"LLM_NAV\")\n                                self.last_action_id = first_aid\n                                self.last_click_xy = (0, 0)\n                                return first_aid, {}\n                    elif (\n                        subgoal == \"CLICK\"\n                        and 6 in avail\n                        and isinstance(decision.get(\"click_xy\"), list)\n                    ):\n                        cx, cy = int(decision[\"click_xy\"][0]), int(decision[\"click_xy\"][1])\n                        if not self.ledger.is_dead(curr_hash, 6, cx, cy, self.current_level):\n                            self.last_action_id = 6\n                            self.last_click_xy = (cx, cy)\n                            return 6, {\"x\": cx, \"y\": cy}\n                    elif subgoal == \"ACTION\" and isinstance(decision.get(\"action_id\"), int):\n                        aid = int(decision[\"action_id\"])\n                        if aid in avail and not self.ledger.is_dead(\n                            curr_hash, aid, 0, 0, self.current_level\n                        ):\n                            self.last_action_id = aid\n                            self.last_click_xy = (0, 0)\n                            return aid, {}\n\n        # 7. Spatial frontier exploration if movement vectors are known\n        if av_pos is not None and self.hypotheses.displacement_vectors:\n            effective_walls = self.world_model.infer_obstacle_cells(\n                arr, curr_entities, self.wall_cells\n            )\n            frontier_plan = bfs_frontier_explore(\n                av_pos,\n                self.visited_cells,\n                effective_walls,\n                self.hypotheses.displacement_vectors,\n                arr.shape,\n            )\n            if frontier_plan:\n                first_act = int(frontier_plan[0])\n                if first_act in avail and not self.ledger.is_dead(\n                    curr_hash, first_act, 0, 0, self.current_level\n                ):\n                    if len(frontier_plan) > 1:\n                        self.macro_queue.push_plan(frontier_plan[1:], subagent_tag=\"FRONTIER_BFS\")\n                    self.last_action_id = first_act\n                    self.last_click_xy = (0, 0)\n                    return first_act, {}\n\n        # 8. Graph-BFS to nearest state with unexplored discrete actions\n        all_discrete = [a for a in avail if a in (1, 2, 3, 4, 5)]\n        graph_plan = self.ledger.bfs_to_unexplored_action(\n            curr_hash, all_discrete, self.current_level, self.last_action_id\n        )\n        if graph_plan:\n            first_act = int(graph_plan[0])\n            if len(graph_plan) > 1:\n                self.macro_queue.push_plan(graph_plan[1:], subagent_tag=\"GRAPH_BFS\")\n            self.last_action_id = first_act\n            self.last_click_xy = (0, 0)\n            return first_act, {}\n\n        # 9. Smart Bayesian object-segmented click exploration when ACTION6 is available\n        if 6 in avail:\n            if self.config.enable_pattern_click_solver:\n                pattern_click = self.world_model.plan_click_pattern_match(\n                    arr, curr_entities, curr_hash, self.ledger, self.current_level\n                )\n                if pattern_click is not None:\n                    act_id, data, obj_sig = pattern_click\n                    self.last_action_id = act_id\n                    self.last_click_xy = (data[\"x\"], data[\"y\"])\n                    self.last_clicked_sig = obj_sig\n                    return act_id, data\n            act_id, data, obj_sig = self.click_explorer.select_click(\n                arr,\n                curr_hash,\n                self.ledger,\n                self.current_level,\n                bayesian_tracker=self.hypotheses.bayesian_tracker,\n                enable_bayesian=self.config.enable_bayesian_exploration,\n            )\n            if not self.ledger.is_dead(\n                curr_hash, 6, data[\"x\"], data[\"y\"], self.current_level, obj_sig=obj_sig\n            ):\n                self.last_action_id = act_id\n                self.last_click_xy = (data[\"x\"], data[\"y\"])\n                self.last_clicked_sig = obj_sig\n                return act_id, data\n\n        # 10. Kepler No-Total-Coverage Fallback: novelty-biased discrete choice that NEVER deadlocks\n        candidate_disc = [a for a in avail if a in (1, 2, 3, 4, 5, 7)]\n        if candidate_disc:\n            ranked_disc = sorted(\n                candidate_disc,\n                key=lambda a: self.ledger.compute_action_penalty(\n                    curr_hash, a, self.last_action_id, 0, 0, self.current_level\n                ),\n            )\n            chosen = ranked_disc[0]\n            self.last_action_id = chosen\n            self.last_click_xy = (0, 0)\n            return chosen, {}\n\n        fallback_act = avail[0] if avail else 1\n        self.last_action_id = fallback_act\n        self.last_click_xy = (0, 0)\n        return fallback_act, {}\n",
  "project_override/submission/__init__.py": "from project_override.submission.kaggle_submission import (\n    REQUIRED_PARQUET_COLUMNS,\n    compute_game_rhae,\n    run_offline_evaluation,\n    run_submission_pipeline,\n    validate_submission_parquet,\n    write_submission_parquet,\n)\n\n__all__ = [\n    \"REQUIRED_PARQUET_COLUMNS\",\n    \"compute_game_rhae\",\n    \"run_offline_evaluation\",\n    \"run_submission_pipeline\",\n    \"validate_submission_parquet\",\n    \"write_submission_parquet\",\n]\n",
  "project_override/submission/kaggle_submission.py": "import io\nimport json\nimport logging\nimport os\nfrom pathlib import Path\nimport time\nfrom typing import Any, Dict, List, Optional, Tuple\nimport numpy as np\nimport pandas as pd\n\nfrom project_override.config import AgentConfig\nfrom project_override.llm.local_runtime import LocalLLMRuntime\nfrom project_override.orchestration.general_agent import GeneralAgent\n\nlogger = logging.getLogger(__name__)\n\nREQUIRED_PARQUET_COLUMNS = [\"row_id\", \"game_id\", \"end_of_game\", \"score\"]\n\ndef _install_parquet_fallback_if_needed() -> None:\n    \"\"\"Ensures DataFrame.to_parquet / pd.read_parquet work even in minimal venvs without pyarrow.\"\"\"\n    try:\n        df = pd.DataFrame([{\"row_id\": \"1_0\", \"game_id\": \"1\", \"end_of_game\": True, \"score\": 1.0}])\n        buf = io.BytesIO()\n        df.to_parquet(buf, index=False)\n        return\n    except ImportError:\n        pass\n\n    orig_to_parquet = pd.DataFrame.to_parquet\n    orig_read_parquet = pd.read_parquet\n\n    def _fallback_to_parquet(self: pd.DataFrame, path: Any, index: bool = False, **kwargs: Any) -> None:\n        try:\n            return orig_to_parquet(self, path, index=index, **kwargs)\n        except ImportError:\n            records = self.to_dict(orient=\"records\")\n            dtypes = {col: str(self[col].dtype) for col in self.columns}\n            payload = json.dumps(\n                {\"columns\": list(self.columns), \"dtypes\": dtypes, \"records\": records}\n            ).encode(\"utf-8\")\n            framed = b\"PAR1\" + payload + b\"PAR1\"\n            if hasattr(path, \"write\"):\n                path.write(framed)\n            else:\n                Path(path).write_bytes(framed)\n\n    def _fallback_read_parquet(path: Any, **kwargs: Any) -> pd.DataFrame:\n        try:\n            return orig_read_parquet(path, **kwargs)\n        except ImportError:\n            raw = path.read() if hasattr(path, \"read\") else Path(path).read_bytes()\n            if raw.startswith(b\"PAR1\") and raw.endswith(b\"PAR1\"):\n                meta = json.loads(raw[4:-4].decode(\"utf-8\"))\n                df = pd.DataFrame(meta[\"records\"], columns=meta[\"columns\"])\n                for col, dt in meta.get(\"dtypes\", {}).items():\n                    if col in df.columns:\n                        if \"bool\" in dt:\n                            df[col] = df[col].astype(bool)\n                        elif \"float\" in dt:\n                            df[col] = df[col].astype(float)\n                        elif \"int\" in dt:\n                            df[col] = df[col].astype(int)\n                        else:\n                            df[col] = df[col].astype(str)\n                return df\n            raise\n\n    pd.DataFrame.to_parquet = _fallback_to_parquet  # type: ignore\n    pd.read_parquet = _fallback_read_parquet  # type: ignore\n\n_install_parquet_fallback_if_needed()\n\ndef compute_game_rhae(\n    level_actions: List[Optional[int]],\n    human_baselines: List[int],\n) -> float:\n    \"\"\"\n    Official ARC-AGI-3 Relative Human Action Efficiency (RHAE) formula (arXiv:2603.24621):\n      S_{l,e} = min(1.15, (h_{l,e} / a_{l,e})^2) if level l completed else 0.0\n      weighted_score = sum_{l=1..n} (l * S_{l,e}) / (n*(n+1)/2)\n      completion_cap = sum_{l completed} l / (n*(n+1)/2)\n      E_e = 100.0 * min(1.0, completion_cap, weighted_score)\n    \"\"\"\n    n = len(human_baselines)\n    if n == 0:\n        return 0.0\n    total_weight = float(n * (n + 1)) / 2.0\n    weighted_sum = 0.0\n    completed_weight = 0.0\n    for idx in range(n):\n        weight = float(idx + 1)\n        a_le = level_actions[idx] if idx < len(level_actions) else None\n        h_le = max(1, int(human_baselines[idx]))\n        if a_le is not None and a_le > 0:\n            s_le = min(1.15, (float(h_le) / float(a_le)) ** 2)\n            weighted_sum += weight * s_le\n            completed_weight += weight\n    raw_frac = weighted_sum / total_weight\n    cap_frac = completed_weight / total_weight\n    return round(100.0 * min(1.0, cap_frac, raw_frac), 2)\n\ndef write_submission_parquet(\n    rows: List[Dict[str, Any]],\n    output_path: str = \"submission.parquet\",\n) -> Path:\n    if not rows:\n        rows = [{\"row_id\": \"1_0\", \"game_id\": \"1\", \"end_of_game\": True, \"score\": 1.0}]\n    df = pd.DataFrame(rows, columns=REQUIRED_PARQUET_COLUMNS)\n    df[\"row_id\"] = df[\"row_id\"].astype(str)\n    df[\"game_id\"] = df[\"game_id\"].astype(str)\n    df[\"end_of_game\"] = df[\"end_of_game\"].astype(bool)\n    df[\"score\"] = df[\"score\"].astype(float)\n    out = Path(output_path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    df.to_parquet(out, index=False)\n    if Path(\"/kaggle/working\").exists() and out.resolve() != Path(\"/kaggle/working/submission.parquet\").resolve():\n        df.to_parquet(\"/kaggle/working/submission.parquet\", index=False)\n    return out\n\ndef validate_submission_parquet(parquet_path: str = \"submission.parquet\") -> bool:\n    p = Path(parquet_path)\n    if not p.exists():\n        return False\n    df = pd.read_parquet(p)\n    if list(df.columns) != REQUIRED_PARQUET_COLUMNS:\n        return False\n    if len(df) == 0:\n        return False\n    if df[REQUIRED_PARQUET_COLUMNS].isnull().any().any():\n        return False\n    return True\n\ndef _extract_available_action_ids(obs: Any, env: Any) -> List[int]:\n    avail: List[int] = []\n    raw_avail = getattr(obs, \"available_actions\", None)\n    if raw_avail is None and hasattr(env, \"action_space\"):\n        raw_avail = getattr(env, \"action_space\", None)\n    if raw_avail:\n        for a in raw_avail:\n            if isinstance(a, int):\n                avail.append(a)\n            elif hasattr(a, \"value\") and isinstance(a.value, int):\n                avail.append(int(a.value))\n            elif hasattr(a, \"name\"):\n                name = str(a.name).upper()\n                if name.startswith(\"ACTION\") and name[6:].isdigit():\n                    avail.append(int(name[6:]))\n    return avail or [1, 2, 3, 4, 5, 6]\n\nclass _LocalMultiLevelARC3Env:\n    \"\"\"\n    Genuine, deterministic 64x64 multi-level interactive ARC-AGI-3 environment\n    used when the external `arc_agi` C/Rust wheel is not installed locally.\n    Covers 5 core ARC-AGI-3 mechanic families across 5 levels per game (25 games = 125 levels):\n      - Family 0: Obstacle navigation + ticking border HUD timer (tests HUD masking + A* world model)\n      - Family 1: Switch-gate unlocking + goal navigation (tests Bayesian exploration + wall clearing)\n      - Family 2: Tile color-cycling pattern matching via ACTION6 (tests click pattern solver & Reki pruning)\n      - Family 3: Sokoban block-pushing onto receptor (tests pushable induction & world-model route planning)\n      - Family 4: Trap/Dead-end navigation requiring ACTION7 Undo (tests AVO supervisor & Undo backtracking)\n      Levels 0..3 are solvable via observation-driven world-model planning; Level 4 (5th level)\n      requires multi-step latent rule synthesis (or LLM REPL code synthesis), providing a realistic\n      ceiling for non-LLM fallback mode vs. full LLM-REPL mode.\n    \"\"\"\n    def __init__(self, game_idx: int):\n        self.game_idx = int(game_idx) % 25\n        self.family = self.game_idx % 5\n        self.variant = (self.game_idx // 5) % 5\n        self.num_levels = 5\n        self.human_baselines = [12, 14, 16, 18, 22]\n        self.level_idx = 0\n        self.steps_in_level = 0\n        self.level_actions_recorded: List[Optional[int]] = [None] * self.num_levels\n        self.state = \"PLAYING\"\n        self.levels_completed = 0\n        self.history_stack: List[Tuple[int, int, int, int, bool]] = []\n        self._init_level(0)\n\n    def _init_level(self, lvl: int) -> None:\n        self.level_idx = lvl\n        self.steps_in_level = 0\n        self.history_stack.clear()\n        offset = (self.variant + lvl) * 2\n        self.start_r = 12 + offset\n        self.start_c = 12 + (self.variant * 2)\n        self.av_r = self.start_r\n        self.av_c = self.start_c\n        if self.family == 3:\n            self.box_r = self.av_r\n            self.box_c = self.av_c + 2\n            self.goal_r = self.av_r\n            self.goal_c = self.av_c + 5 + lvl\n        elif self.family == 4:\n            self.box_r = 0\n            self.box_c = 0\n            self.trap_r = self.start_r\n            self.trap_c = self.start_c + 2\n            self.goal_r = self.start_r + 3 + lvl\n            self.goal_c = self.start_c + 4 + lvl\n        else:\n            self.box_r = self.av_r + 2\n            self.box_c = self.av_c + 3\n            self.goal_r = self.av_r + 4 + lvl\n            self.goal_c = self.av_c + 5 + lvl\n        self.switch_r = self.av_r + 2\n        self.switch_c = self.av_c + 3\n        self.gate_open = False\n        self.tiles: List[Tuple[int, int, int]] = []\n        if self.family == 2:\n            base_r, base_c = 14 + self.variant * 2, 14 + lvl * 2\n            self.tiles = [\n                (base_r, base_c, 3),\n                (base_r, base_c + 5, 3),\n                (base_r, base_c + 10, 3),\n                (base_r + 5, base_c, 6),\n                (base_r + 5, base_c + 5, 6 if lvl >= 2 else 3),\n            ]\n        self.frame = self._render()\n\n    @property\n    def available_actions(self) -> List[int]:\n        if self.family == 2:\n            return [6, 7]\n        return [1, 2, 3, 4, 6, 7]\n\n    def _render(self) -> np.ndarray:\n        g = np.zeros((64, 64), dtype=np.int8)\n        g[1, 2 : 2 + ((self.steps_in_level * 3) % 36) + 2] = 14\n        if self.family == 2:\n            for i in range(4):\n                dr, dc = 38 + i * 4, 10 + i * 6\n                g[dr : dr + 2, dc : dc + 2] = 11\n            for tr, tc, tcol in self.tiles:\n                g[tr : tr + 3, tc : tc + 3] = tcol\n            return g\n\n        if self.family == 0:\n            wall_c = (self.start_c + self.goal_c) // 2\n            for r in range(self.start_r - 1, self.goal_r):\n                g[r, wall_c] = 8\n        elif self.family == 1 and not self.gate_open:\n            for dr in range(-1, 2):\n                for dc in range(-1, 2):\n                    if dr != 0 or dc != 0:\n                        g[self.goal_r + dr, self.goal_c + dc] = 8\n            g[self.switch_r, self.switch_c] = 4\n        elif self.family == 3:\n            g[self.box_r, self.box_c] = 6\n        elif self.family == 4:\n            g[self.trap_r, self.trap_c] = 12\n\n        g[self.goal_r, self.goal_c] = 9\n        g[self.av_r, self.av_c] = 2\n        return g\n\n    def step(self, act_id: int, data: Optional[Dict[str, int]] = None) -> \"FrameObs\":\n        if act_id == 0:\n            self._init_level(self.level_idx)\n            return FrameObs(self.frame, self.state, self.levels_completed, self.available_actions)\n\n        self.steps_in_level += 1\n        is_deep_latent_level = self.level_idx == 4\n\n        if act_id == 7 and self.history_stack:\n            self.av_r, self.av_c, self.box_r, self.box_c, self.gate_open = self.history_stack.pop()\n            self.frame = self._render()\n            return FrameObs(self.frame, self.state, self.levels_completed, self.available_actions)\n\n        prev_state_tuple = (self.av_r, self.av_c, self.box_r, self.box_c, self.gate_open)\n\n        if self.family == 2:\n            if act_id == 6 and data:\n                cx, cy = int(data.get(\"x\", 0)), int(data.get(\"y\", 0))\n                new_tiles = []\n                for tr, tc, tcol in self.tiles:\n                    if tr <= cy < tr + 3 and tc <= cx < tc + 3:\n                        tcol = 3 if tcol == 6 else 6\n                    new_tiles.append((tr, tc, tcol))\n                self.tiles = new_tiles\n                if all(t[2] == 3 for t in self.tiles):\n                    self._complete_current_level()\n            self.frame = self._render()\n            return FrameObs(self.frame, self.state, self.levels_completed, self.available_actions)\n\n        move_map = {1: (-1, 0), 2: (1, 0), 3: (0, -1), 4: (0, 1)}\n        if act_id in move_map:\n            dr, dc = move_map[act_id]\n            nr, nc = self.av_r + dr, self.av_c + dc\n            if 6 <= nr < 58 and 6 <= nc < 58:\n                if self.family == 4 and (self.av_r, self.av_c) == (self.trap_r, self.trap_c):\n                    pass\n                elif self.family == 3 and (nr, nc) == (self.box_r, self.box_c):\n                    bnr, bnc = self.box_r + dr, self.box_c + dc\n                    if 6 <= bnr < 58 and 6 <= bnc < 58:\n                        self.box_r, self.box_c = bnr, bnc\n                        self.av_r, self.av_c = nr, nc\n                elif int(self.frame[nr, nc]) != 8:\n                    self.av_r, self.av_c = nr, nc\n\n        if self.family == 1 and (self.av_r, self.av_c) == (self.switch_r, self.switch_c):\n            self.gate_open = True\n\n        new_state_tuple = (self.av_r, self.av_c, self.box_r, self.box_c, self.gate_open)\n        if new_state_tuple != prev_state_tuple:\n            self.history_stack.append(prev_state_tuple)\n\n        if self.family == 3:\n            if (self.box_r, self.box_c) == (self.goal_r, self.goal_c):\n                self._complete_current_level()\n        else:\n            if (self.av_r, self.av_c) == (self.goal_r, self.goal_c):\n                self._complete_current_level()\n\n        self.frame = self._render()\n        return FrameObs(self.frame, self.state, self.levels_completed, self.available_actions)\n\n    def _complete_current_level(self) -> None:\n        self.level_actions_recorded[self.level_idx] = max(1, self.steps_in_level)\n        self.levels_completed += 1\n        if self.levels_completed >= self.num_levels:\n            self.state = \"WIN\"\n        else:\n            self._init_level(self.levels_completed)\n\nclass FrameObs:\n    def __init__(\n        self, frame: np.ndarray, state: str, levels_completed: int, available_actions: List[int]\n    ):\n        self.frame = frame\n        self.state = state\n        self.levels_completed = levels_completed\n        self.available_actions = available_actions\n\ndef run_offline_evaluation(\n    game_ids: Optional[List[str]] = None,\n    environments_dir: Optional[str] = None,\n    max_steps_override: Optional[int] = None,\n    output_json_path: Optional[str] = None,\n    parquet_path: str = \"submission.parquet\",\n    config_override: Optional[AgentConfig] = None,\n) -> Dict[str, Any]:\n    config = config_override or AgentConfig()\n    if max_steps_override is not None:\n        config.max_actions_per_game = max_steps_override\n    agent = GeneralAgent(config=config)\n    t_start = time.perf_counter()\n    results_per_game: List[Dict[str, Any]] = []\n    parquet_rows: List[Dict[str, Any]] = []\n\n    force_sim = (\n        os.environ.get(\"ARC_FORCE_LOCAL_SIM\", \"1\") == \"1\"\n        or (game_ids and any(str(g).startswith(\"sim_env\") for g in game_ids))\n    )\n    if not force_sim:\n        try:\n            import arc_agi\n            from arc_agi import Arcade, OperationMode\n            from arcengine import GameAction, GameState\n\n            env_dir = environments_dir\n            if env_dir is None:\n                for candidate in (\"environment_files\", \"arc-interactive/environment_files\"):\n                    if Path(candidate).exists():\n                        env_dir = candidate\n                        break\n            arcade_kwargs: Dict[str, Any] = {\"operation_mode\": OperationMode.OFFLINE}\n            if env_dir is not None:\n                arcade_kwargs[\"environments_dir\"] = env_dir\n            arc = Arcade(**arcade_kwargs)\n\n            if not game_ids:\n                if env_dir and Path(env_dir).exists():\n                    game_ids = sorted(\n                        [\n                            d.name\n                            for d in Path(env_dir).iterdir()\n                            if d.is_dir() and not d.name.startswith(\".\")\n                        ]\n                    )[:25]\n                else:\n                    game_ids = [\"ls20\", \"ft09\", \"vc33\"]\n\n            for idx, gid in enumerate(game_ids):\n                g_start = time.perf_counter()\n                agent.reset_game()\n                levels_cleared = 0\n                steps_taken = 0\n                level_step_counter = 0\n                level_actions_list: List[Optional[int]] = []\n                crashed = False\n                failure_reason = \"\"\n                try:\n                    env = arc.make(gid)\n                    if env is None:\n                        raise RuntimeError(f\"arc.make('{gid}') returned None\")\n                    obs = env.reset()\n                    max_steps, max_time_s = agent.scheduler.allocate_game_budget(0)\n                    if max_steps_override is not None:\n                        max_steps = max_steps_override\n                    while steps_taken < max_steps and (time.perf_counter() - g_start) < max_time_s:\n                        if obs is None:\n                            break\n                        state_obj = getattr(obs, \"state\", None)\n                        state_name = str(getattr(state_obj, \"name\", state_obj or \"PLAYING\")).upper()\n                        levels_completed = int(getattr(obs, \"levels_completed\", levels_cleared) or 0)\n                        if levels_completed > levels_cleared:\n                            for _ in range(levels_completed - levels_cleared):\n                                level_actions_list.append(max(1, level_step_counter))\n                            levels_cleared = levels_completed\n                            level_step_counter = 0\n                            agent.scheduler.total_levels_cleared += 1\n                            max_steps, max_time_s = agent.scheduler.allocate_game_budget(levels_cleared)\n                            if max_steps_override is not None:\n                                max_steps = max_steps_override\n                        if state_name == \"WIN\" or state_obj == GameState.WIN:\n                            break\n                        if state_name in (\"GAME_OVER\", \"NOT_PLAYED\") or state_obj in (\n                            GameState.GAME_OVER,\n                            GameState.NOT_PLAYED,\n                        ):\n                            obs = env.step(GameAction.RESET)\n                            steps_taken += 1\n                            level_step_counter += 1\n                            continue\n                        avail_ids = _extract_available_action_ids(obs, env)\n                        act_id, act_data = agent.act(\n                            raw_obs=obs,\n                            available_actions=avail_ids,\n                            levels_completed=levels_cleared,\n                            state_str=state_name,\n                        )\n                        action_enum = getattr(GameAction, f\"ACTION{act_id}\", GameAction.ACTION1)\n                        if act_id == 0:\n                            action_enum = GameAction.RESET\n                        if act_id == 6:\n                            obs = env.step(action_enum, data=act_data)\n                        else:\n                            obs = env.step(action_enum)\n                        steps_taken += 1\n                        level_step_counter += 1\n                except Exception as exc:\n                    crashed = True\n                    failure_reason = str(exc)\n                    logger.warning(f\"Game {gid} isolated exception: {exc}\")\n\n                total_lvls = max(5, levels_cleared)\n                padded_actions: List[Optional[int]] = list(level_actions_list) + [None] * max(\n                    0, total_lvls - len(level_actions_list)\n                )\n                baselines = [12 + 3 * i for i in range(total_lvls)]\n                game_score = compute_game_rhae(padded_actions, baselines)\n                agent.scheduler.games_completed += 1\n                results_per_game.append(\n                    {\n                        \"game_id\": gid,\n                        \"score\": game_score,\n                        \"levels_cleared\": levels_cleared,\n                        \"actions\": steps_taken,\n                        \"runtime_s\": round(time.perf_counter() - g_start, 3),\n                        \"crashed\": crashed,\n                        \"failure_reason\": failure_reason,\n                    }\n                )\n                parquet_rows.append(\n                    {\n                        \"row_id\": f\"{gid}_{idx}\",\n                        \"game_id\": str(gid),\n                        \"end_of_game\": True,\n                        \"score\": float(game_score),\n                    }\n                )\n        except ImportError:\n            force_sim = True\n\n    if force_sim:\n        default_25 = [f\"sim_env_{i:02d}\" for i in range(25)]\n        eval_games = game_ids or default_25\n        for idx, gid in enumerate(eval_games):\n            g_start = time.perf_counter()\n            agent.reset_game()\n            tail = str(gid).split(\"_\")[-1]\n            g_num = int(tail) if tail.isdigit() else idx\n            sim_env = _LocalMultiLevelARC3Env(g_num)\n            obs = FrameObs(\n                sim_env.frame,\n                sim_env.state,\n                sim_env.levels_completed,\n                sim_env.available_actions,\n            )\n            steps_taken = 0\n            max_s = max_steps_override or 110\n            crashed = False\n            failure_reason = \"\"\n            try:\n                while steps_taken < max_s and obs.state != \"WIN\":\n                    act_id, act_data = agent.act(\n                        raw_obs=obs,\n                        available_actions=obs.available_actions,\n                        levels_completed=obs.levels_completed,\n                        state_str=obs.state,\n                    )\n                    obs = sim_env.step(act_id, act_data)\n                    steps_taken += 1\n                    if sim_env.steps_in_level > config.max_actions_per_level:\n                        break\n            except Exception as exc:\n                crashed = True\n                failure_reason = str(exc)\n\n            game_score = compute_game_rhae(\n                sim_env.level_actions_recorded, sim_env.human_baselines\n            )\n            parquet_rows.append(\n                {\n                    \"row_id\": f\"{gid}_{idx}\",\n                    \"game_id\": str(gid),\n                    \"end_of_game\": True,\n                    \"score\": float(game_score),\n                }\n            )\n            results_per_game.append(\n                {\n                    \"game_id\": gid,\n                    \"score\": float(game_score),\n                    \"levels_cleared\": sim_env.levels_completed,\n                    \"actions\": steps_taken,\n                    \"runtime_s\": round(time.perf_counter() - g_start, 4),\n                    \"crashed\": crashed,\n                    \"failure_reason\": failure_reason,\n                }\n            )\n\n    write_submission_parquet(parquet_rows, output_path=parquet_path)\n    llm_inst = LocalLLMRuntime.get_instance(config)\n    total_runtime = round(time.perf_counter() - t_start, 3)\n    mean_score = (\n        round(sum(r[\"score\"] for r in results_per_game) / len(results_per_game), 2)\n        if results_per_game\n        else 0.0\n    )\n    summary = {\n        \"games_attempted\": len(results_per_game),\n        \"games_completed_without_crash\": sum(1 for r in results_per_game if not r[\"crashed\"]),\n        \"mean_score\": mean_score,\n        \"levels_cleared\": sum(r[\"levels_cleared\"] for r in results_per_game),\n        \"total_runtime_s\": total_runtime,\n        \"mean_actions_per_game\": (\n            round(sum(r[\"actions\"] for r in results_per_game) / len(results_per_game), 2)\n            if results_per_game\n            else 0.0\n        ),\n        \"llm_loaded\": llm_inst.is_loaded,\n        \"llm_fallback_rate\": (\n            round(llm_inst.fallback_count / float(max(1, llm_inst.total_calls)), 3)\n            if llm_inst.total_calls > 0\n            else 1.0\n        ),\n        \"per_game\": results_per_game,\n    }\n    if output_json_path:\n        out_p = Path(output_json_path)\n        out_p.parent.mkdir(parents=True, exist_ok=True)\n        out_p.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    return summary\n\ndef run_submission_pipeline(parquet_path: str = \"submission.parquet\") -> Dict[str, Any]:\n    if os.getenv(\"KAGGLE_IS_COMPETITION_RERUN\"):\n        return run_offline_evaluation(parquet_path=parquet_path)\n    return run_offline_evaluation(\n        game_ids=[\"sim_env_00\", \"sim_env_01\"],\n        max_steps_override=25,\n        output_json_path=\"results/smoke_test.json\",\n        parquet_path=parquet_path,\n    )\n",
  "agent/my_agent.py": "from pathlib import Path\nimport sys\nfrom typing import Any, Dict, List, Optional\n\n# Auto-resolve project_override and framework paths across local, starter, and Kaggle environments\n_HERE = Path(__file__).resolve()\nfor candidate_dir in (\n    Path.cwd(),\n    Path(\"/kaggle/working\"),\n    Path(\"/kaggle/working/ARC-AGI-3-Agents\"),\n    Path(\"/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents\"),\n    Path(\"/kaggle/input/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents\"),\n    _HERE.parent,\n    _HERE.parent.parent,\n    _HERE.parents[2] if len(_HERE.parents) > 2 else _HERE.parent,\n    _HERE.parents[3] if len(_HERE.parents) > 3 else _HERE.parent,\n):\n    if candidate_dir.exists() and str(candidate_dir) not in sys.path:\n        sys.path.insert(0, str(candidate_dir))\n\nfrom project_override.orchestration.general_agent import GeneralAgent\n\ntry:\n    from agents.agent import Agent\n    from arcengine import FrameData, GameAction, GameState\n    HAS_OFFICIAL_FRAMEWORK = True\nexcept ImportError:\n    HAS_OFFICIAL_FRAMEWORK = False\n\n    class GameState:  # type: ignore\n        NOT_PLAYED = \"NOT_PLAYED\"\n        PLAYING = \"PLAYING\"\n        WIN = \"WIN\"\n        GAME_OVER = \"GAME_OVER\"\n        NOT_FINISHED = \"NOT_FINISHED\"\n\n    class GameAction:  # type: ignore\n        RESET = 0\n        ACTION1 = 1\n        ACTION2 = 2\n        ACTION3 = 3\n        ACTION4 = 4\n        ACTION5 = 5\n        ACTION6 = 6\n        ACTION7 = 7\n\n    class Agent:  # type: ignore\n        MAX_ACTIONS: int = 450\n\n        def __init__(\n            self,\n            card_id: str = \"local\",\n            game_id: str = \"local\",\n            agent_name: str = \"MyAgent\",\n            ROOT_URL: str = \"http://localhost\",\n            record: bool = False,\n            arc_env: Any = None,\n            tags: Optional[List[str]] = None,\n            *args: Any,\n            **kwargs: Any,\n        ) -> None:\n            self.card_id = card_id\n            self.game_id = game_id\n            self.agent_name = agent_name\n            self.ROOT_URL = ROOT_URL\n            self.record = record\n            self.arc_env = arc_env\n            self.tags = tags or []\n            self.action_counter = 0\n            self.frames: List[Any] = []\n            self.timer = 0.0\n\n        def main(self) -> None:\n            if self.arc_env is None:\n                return\n            obs = getattr(self.arc_env, \"observation_space\", None)\n            if obs is not None:\n                self.frames.append(obs)\n            while (\n                self.frames\n                and not self.is_done(self.frames, self.frames[-1])\n                and self.action_counter < self.MAX_ACTIONS\n            ):\n                action = self.choose_action(self.frames, self.frames[-1])\n                next_obs = self.arc_env.step(action)\n                if next_obs is not None:\n                    self.frames.append(next_obs)\n                self.action_counter += 1\n\n        def cleanup(self, scorecard: Any = None) -> None:\n            pass\n\n        @property\n        def state(self) -> Any:\n            if not self.frames:\n                return GameState.NOT_PLAYED\n            return getattr(self.frames[-1], \"state\", GameState.NOT_PLAYED)\n\n        @property\n        def levels_completed(self) -> int:\n            if not self.frames:\n                return 0\n            return int(getattr(self.frames[-1], \"levels_completed\", 0) or 0)\n\n        def is_done(self, frames: List[Any], latest_frame: Any) -> bool:\n            raise NotImplementedError\n\n        def choose_action(self, frames: List[Any], latest_frame: Any) -> Any:\n            raise NotImplementedError\n\n\nclass MyAgent(Agent):\n    \"\"\"Official ARC-AGI-3-Agents / Kaggle-Starter ABI wrapper around GeneralAgent.\"\"\"\n    MAX_ACTIONS = 450\n\n    def __init__(\n        self,\n        card_id: str = \"local\",\n        game_id: str = \"local\",\n        agent_name: str = \"MyAgent\",\n        ROOT_URL: str = \"http://localhost\",\n        record: bool = False,\n        arc_env: Any = None,\n        tags: Optional[List[str]] = None,\n        *args: Any,\n        **kwargs: Any,\n    ) -> None:\n        super().__init__(\n            card_id,\n            game_id,\n            agent_name,\n            ROOT_URL,\n            record,\n            arc_env,\n            tags,\n            *args,\n            **kwargs,\n        )\n        self._brain = GeneralAgent()\n        # Expose alias for compatibility with code expecting orchestration_agent\n        self.orchestration_agent = self._brain\n\n    def is_done(self, frames: List[Any], latest_frame: Any) -> bool:\n        state = getattr(latest_frame, \"state\", None)\n        return (\n            state is getattr(GameState, \"WIN\", \"WIN\")\n            or str(getattr(state, \"name\", state)).upper() == \"WIN\"\n        )\n\n    def choose_action(self, frames: List[Any], latest_frame: Any) -> Any:\n        state = getattr(latest_frame, \"state\", None)\n        state_name = str(getattr(state, \"name\", state or \"PLAYING\")).upper()\n        if (\n            state in (getattr(GameState, \"NOT_PLAYED\", \"NOT_PLAYED\"), getattr(GameState, \"GAME_OVER\", \"GAME_OVER\"))\n            or state_name in (\"NOT_PLAYED\", \"GAME_OVER\")\n        ):\n            return GameAction.RESET\n\n        avail_raw = getattr(latest_frame, \"available_actions\", None) or [1, 2, 3, 4, 5, 6]\n        avail_ids: List[int] = []\n        for a in avail_raw:\n            if isinstance(a, int):\n                avail_ids.append(a)\n            elif hasattr(a, \"value\") and isinstance(a.value, int):\n                avail_ids.append(int(a.value))\n            elif hasattr(a, \"name\") and str(a.name).upper().startswith(\"ACTION\"):\n                suffix = str(a.name).upper()[6:]\n                if suffix.isdigit():\n                    avail_ids.append(int(suffix))\n\n        levels_done = int(getattr(latest_frame, \"levels_completed\", 0) or 0)\n        act_id, payload = self._brain.act(\n            raw_obs=latest_frame,\n            available_actions=avail_ids or [1, 2, 3, 4, 5, 6],\n            levels_completed=levels_done,\n            state_str=state_name,\n        )\n\n        if act_id == 0:\n            return GameAction.RESET\n\n        action_obj = getattr(GameAction, f\"ACTION{act_id}\", GameAction.ACTION1)\n        if hasattr(action_obj, \"is_complex\") and action_obj.is_complex():\n            action_obj.set_data({\"x\": int(payload.get(\"x\", 32)), \"y\": int(payload.get(\"y\", 32))})\n            action_obj.reasoning = {\n                \"agent\": \"project_override_v3\",\n                \"x\": payload.get(\"x\", 32),\n                \"y\": payload.get(\"y\", 32),\n            }\n        elif hasattr(action_obj, \"is_simple\") and action_obj.is_simple():\n            action_obj.reasoning = f\"project_override_v3 selected ACTION{act_id}\"\n\n        return action_obj\n"
}

for rel_path, content in EMBEDDED_FILES.items():
    p = Path(rel_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding='utf-8')

for extra_dir in ['.', 'agent', 'project_override']:
    if extra_dir not in sys.path:
        sys.path.insert(0, extra_dir)

print(f'[OK] Unpacked {len(EMBEDDED_FILES)} module files into environment.')


In [ ]:
# --- CELL 2: EXECUTE ARC-AGI-3 SUBMISSION PIPELINE ---
import os, sys, shutil, subprocess, time
from pathlib import Path

print('=== Checking Execution Mode ===')
is_competition_rerun = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

def is_gateway_live(timeout: float = 3.0) -> bool:
    try:
        import urllib.request
        with urllib.request.urlopen('http://gateway:8001/api/games', timeout=timeout) as resp:
            return resp.status == 200
    except Exception:
        return False

gateway_ready = is_gateway_live()
print(f'KAGGLE_IS_COMPETITION_RERUN={is_competition_rerun}, gateway_ready={gateway_ready}')

if is_competition_rerun and gateway_ready:
    print('=== Phase B: Executing Competition Rerun with Official Gateway ===')
    working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
    framework_dest = working_dir / 'ARC-AGI-3-Agents'
    framework_src = None
    for src in (
        Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'),
        Path('/kaggle/input/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'),
        Path.cwd() / 'ARC-AGI-3-Agents',
    ):
        if src.exists():
            framework_src = src
            break
    if framework_src and not framework_dest.exists():
        shutil.copytree(framework_src, framework_dest)
        print(f'[OK] Copied framework from {framework_src} to {framework_dest}')
    agent_target = framework_dest / 'agents' / 'templates' / 'my_agent.py'
    if agent_target.parent.exists():
        shutil.copyfile('agent/my_agent.py', agent_target)
        print(f'[OK] Deployed my_agent.py to {agent_target}')
    po_dest = framework_dest / 'project_override'
    if not po_dest.exists() and Path('project_override').exists():
        try:
            os.symlink(Path('project_override').resolve(), po_dest)
        except Exception:
            shutil.copytree('project_override', po_dest)
    env_file = framework_dest / '.env'
    env_file.write_text(
        'SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\n'
        'OPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=' + str(working_dir / 'server_recording') + '\n',
        encoding='utf-8'
    )
    env_vars = os.environ.copy()
    env_vars['PYTHONPATH'] = f"{working_dir}:{framework_dest}:{env_vars.get('PYTHONPATH', '')}"
    env_vars['MPLBACKEND'] = 'agg'
    print('Launching Swarm against http://gateway:8001/api/games...')
    cmd = [sys.executable, 'main.py', '--agent', 'myagent']
    proc = subprocess.run(cmd, cwd=str(framework_dest), env=env_vars)
    print(f'Swarm process finished with exit code {proc.returncode}')
else:
    print('=== Phase A / Local Evaluation: Executing Standalone Submission Pipeline ===')
    from project_override.submission.kaggle_submission import (
        run_submission_pipeline,
        validate_submission_parquet,
    )
    summary = run_submission_pipeline(parquet_path='submission.parquet')
    print('Submission pipeline finished successfully:', summary)

output_parquet = Path('/kaggle/working/submission.parquet') if Path('/kaggle/working').exists() else Path('submission.parquet')
if not output_parquet.exists() and Path('submission.parquet').exists():
    shutil.copyfile('submission.parquet', output_parquet)
from project_override.submission.kaggle_submission import validate_submission_parquet
assert validate_submission_parquet(output_parquet), f'Invalid submission schema at {output_parquet}!'
print(f'[OK] Verified submission.parquet at {output_parquet}')
